# Module 12 · Expression variability

Senescent cells are described as transcriptionally *noisier*, not just
differently expressed. This module tests that directly.

**The distinction.** Differential expression asks whether the mean moves.
Variability asks whether the spread widens. A gene can have an identical mean in
senescent and non-senescent cells while being far more variable in one — and
increased cell-to-cell heterogeneity is a stated hallmark of senescence, so it
deserves its own test rather than being inferred from the DEG list.

**The confound that governs the design.** The coefficient of variation is
mechanically coupled to sequencing depth and to group size. Senescent cells
carry ~1.9× the UMIs of non-senescent cells and are 20-50× rarer. Both push CV
in a direction that has nothing to do with biology:

- **fewer cells** → noisier variance estimate, generally inflated CV
- **more UMIs** → less sampling noise per cell, generally deflated CV

Sections 03-05 exist to measure both before any CV is compared, and the analysis
runs on a **balanced** subsample for that reason.

**Stratification.** CV is computed twice over, split two ways:

- by **DEG status** — a gene whose mean moved is expected to show altered spread
  as a side effect, so DEGs and non-DEGs must be read separately
- by **HVG status** — highly variable genes are selected *for* variance, so
  including them unstratified would let the selection drive the result

| Section | |
|---|---|
| 01-02 | config, load |
| 03-05 | imbalance assessment, DEG results, depth by senescence |
| 06 | HVGs on the balanced data |
| 07-09 | gene- and donor-level CV, stratified by DEG status |
| 10-13 | gene- and donor-level CV, stratified by HVG status |
| 14-15 | save, summary figures |
| 16-19 | variance partitioning |

**Variance partitioning** (sections 16-19) asks a different question again: of
the total variance in expression, how much is attributable to donor, to
senescence status, to sex, to cohort, and how much is residual. It is the
decomposition that says whether senescence is a large or a small term relative
to the donor effect — and in these data the donor term is usually the big one,
which is the quantitative form of the pseudoreplication argument used
throughout this pipeline.

---
## 01 · Config

**Why.** One dataset registry, one cell type per run. `STUDY_CONFIG` names the
column for every variable the models need, so switching datasets means changing
the key and nothing else.

`SEN_DATA` replaces the hardcoded path root — see `.env.example`.

**Set before running:** `DATASET`, `CELL_TYPE`.

In [ ]:
# -----------------------------------------------------------------------------
# Path resolution - the only addition to the source config
# -----------------------------------------------------------------------------
import os
from pathlib import Path


def _env(name):
    v = os.environ.get(name)
    if not v:
        raise RuntimeError(
            f"{name} is not set. Copy .env.example to .env, edit the paths, "
            f"and source it before starting the kernel.")
    return v.rstrip("/")


SEN_DATA = _env("SENESCENCE_DATA")
print(f"  SENESCENCE_DATA -> {SEN_DATA}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# 08A-1: LIBRARIES
# ════════════════════════════════════════════════════════════════════════════════

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.sparse import issparse

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'font.size': 10, 'axes.labelsize': 10, 'axes.titlesize': 11,
    'legend.fontsize': 9, 'font.family': 'sans-serif',
    'axes.linewidth': 1.0, 'axes.grid': False, 'pdf.fonttype': 42,
})
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=150, dpi_save=300, facecolor='white', frameon=False)

print("✓ Libraries loaded")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# 08A-2: CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

DATASET = "psychad_ad"
CELL_TYPE = "Microglia"

# ─────────────────────────────────────────────────────────────────────────────────
# Dataset Config
# ─────────────────────────────────────────────────────────────────────────────────

STUDY_CONFIG = {
    'psychad_aging': {
        'type': 'aging',
        'primary_var': 'Age',
        'primary_var_type': 'continuous',
        'reference_group': 'Age_40_49',
        'covariates': ['Sex', 'Cohort'],
        'random_effect': 'Sample',
        'sex_col': 'Sex',
        'cohort_col': 'Cohort',
        'cell_type_col': 'subclass',
        'sample_col': 'Sample',
        'senescence_col': 'senescence_label',
        'study_group_col': 'Study_Group',
        'group_order': ['Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59',
                        'Age_60_69', 'Age_70_79', 'Age_80_100'],
    },
    'psychad_ad': {
        'type': 'disease',
        'primary_var': 'Disease_Group',
        'primary_var_type': 'categorical',
        'reference_group': 'Old_Healthy_Control',
        'covariates': ['Age', 'Sex', 'Cohort'],
        'random_effect': 'Sample',
        'sex_col': 'Sex',
        'cohort_col': 'Cohort',
        'cell_type_col': 'subclass',
        'sample_col': 'Sample',
        'senescence_col': 'senescence_label',
        'study_group_col': 'Disease_Group',
        'group_order': ['Young_Healthy_Control', 'Old_Healthy_Control', 'Old_AD'],
    },
    'psychencode': {
        'type': 'aging',
        'primary_var': 'Age_death',
        'primary_var_type': 'continuous',
        'reference_group': 'Age_40_49',
        'covariates': ['Biological_Sex', 'Cohort'],
        'random_effect': 'sample_id',
        'sex_col': 'Biological_Sex',
        'cohort_col': 'Cohort',
        'cell_type_col': 'cell_type',
        'sample_col': 'sample_id',
        'senescence_col': 'senescence_label',
        'study_group_col': 'Study_Group',
        'group_order': ['Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59',
                        'Age_60_69', 'Age_70_79', 'Age_80_100'],
    },
    'blood_itou': {
        'type': 'disease',
        'primary_var': 'disease state',
        'primary_var_type': 'categorical',
        'reference_group': 'Control',
        'covariates': ['Age', 'Sex'],
        'random_effect': 'Sample',
        'sex_col': 'Sex',
        'cohort_col': None,
        'cell_type_col': 'predicted.celltype.l1',
        'sample_col': 'Sample',
        'senescence_col': 'is_senescent',
        'study_group_col': 'disease state',
        'group_order': ['Control', 'Non-rapid ALS', 'Rapid ALS'],
    },
}

config = STUDY_CONFIG[DATASET]
STUDY_TYPE = config['type']
GROUP_ORDER = config['group_order']
CELL_TYPE_COL = config['cell_type_col']
SAMPLE_COL = config['sample_col']
SENESCENCE_COL = config['senescence_col']
STUDY_GROUP_COL = config['study_group_col']
REFERENCE_GROUP = config.get('reference_group', GROUP_ORDER[0])

# ─────────────────────────────────────────────────────────────────────────────────
# Colors
# ─────────────────────────────────────────────────────────────────────────────────

STUDY_GROUP_COLORS = {
    'Age_20_29': '#2E86AB', 'Age_30_39': '#4A90E2', 'Age_40_49': '#50C878',
    'Age_50_59': '#FFB347', 'Age_60_69': '#FF8C00', 'Age_70_79': '#E24A4A',
    'Age_80_100': '#8B0000',
    'Young_Healthy_Control': '#4E79A7', 'Old_Healthy_Control': '#59A14F', 'Old_AD': '#E15759',
    'Control': '#4E79A7', 'MCI': '#F28E2B', 'AD': '#E15759', 'NCI': '#4E79A7',
    'Non-rapid ALS': '#F28E2B', 'Rapid ALS': '#E15759',
}

SNC_COLORS = {"SnC": "#D73027", "Non-SnC": "#BDBDBD"}

CELL_TYPE_COLORS = {
    'Microglia': '#EDC948', 'Astrocyte': '#E15759', 'OPC': '#59A14F',
    'Oligodendrocyte': '#76B7B2', 'Excitatory': '#4E79A7',
    'Inhibitory': '#F28E2B', 'Endothelial': '#B07AA1',
    'Mono': '#4E79A7', 'NK': '#F28E2B', 'B': '#E15759',
    'CD4 T': '#76B7B2', 'CD8 T': '#59A14F',
    'DC': '#B07AA1', 'other T': '#EDC948', 'other': '#BAB0AC',
}

SEX_COLORS = {
    'Male': '#5D6D7E', 'Female': '#A569BD',
    'M': '#5D6D7E', 'F': '#A569BD',
}

# ─────────────────────────────────────────────────────────────────────────────────
# Paths
# ─────────────────────────────────────────────────────────────────────────────────

BASE_DIR = Path(SEN_DATA)

SC_FILE = BASE_DIR / "data" / "04_subsetting" / DATASET / f"{DATASET}_celltypes_subset.h5ad"

DEG_RESULTS_DIR = BASE_DIR / "results" / "05_deg" / DATASET / CELL_TYPE.lower()
RESULTS_DIR = BASE_DIR / "results" / "08_variability" / DATASET / CELL_TYPE.lower()
FIGURES_DIR = BASE_DIR / "figures" / "08_variability" / DATASET / CELL_TYPE.lower()

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────────
# Thresholds
# ─────────────────────────────────────────────────────────────────────────────────

FDR_THRESHOLD = 0.05
LFC_THRESHOLD = 0.5

# ─────────────────────────────────────────────────────────────────────────────────
# Print
# ─────────────────────────────────────────────────────────────────────────────────

print("=" * 80)
print(f"Dataset:        {DATASET}")
print(f"Cell type:      {CELL_TYPE}")
print(f"Study type:     {STUDY_TYPE}")
print(f"Reference:      {REFERENCE_GROUP}")
print(f"Group order:    {GROUP_ORDER}")
print(f"SC file:        {SC_FILE.name}")
print(f"DEG results:    {DEG_RESULTS_DIR}")
print(f"Results:        {RESULTS_DIR}")
print(f"Figures:        {FIGURES_DIR}")
print(f"Senescence col: {SENESCENCE_COL}")
print(f"Cell type col:  {CELL_TYPE_COL}")
print(f"Sample col:     {SAMPLE_COL}")
print("=" * 80)

---
## 02 · Load

**Why.** Reads the scored object from module 03 and subsets to `CELL_TYPE`.
Everything below is within one cell type — CV pooled across cell types would be
dominated by between-type differences in expression level.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# LOAD DATA
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("LOAD DATA")
print("=" * 60)

adata_all = sc.read_h5ad(SC_FILE)
print(f"  Total cells: {adata_all.n_obs:,} x {adata_all.n_vars:,} genes")

# Subset to cell type
adata = adata_all[adata_all.obs[CELL_TYPE_COL] == CELL_TYPE].copy()
print(f"  {CELL_TYPE} cells: {adata.n_obs:,}")

# Ensure senescence column is boolean
if adata.obs[SENESCENCE_COL].dtype == 'object' or adata.obs[SENESCENCE_COL].dtype.name == 'category':
    adata.obs['is_senescent'] = adata.obs[SENESCENCE_COL].map(
        {1: True, 0: False, '1': True, '0': False, True: True, False: False,
         'True': True, 'False': False, 'SnC': True, 'Non-SnC': False}
    ).fillna(False).astype(bool)
else:
    adata.obs['is_senescent'] = adata.obs[SENESCENCE_COL].astype(bool)

n_snc = adata.obs['is_senescent'].sum()
n_nonsnc = (~adata.obs['is_senescent']).sum()
print(f"\n  SnC:     {n_snc:,} ({100*n_snc/len(adata):.1f}%)")
print(f"  Non-SnC: {n_nonsnc:,} ({100*n_nonsnc/len(adata):.1f}%)")
print(f"  Imbalance ratio: {n_nonsnc/max(n_snc,1):.1f}x")

# Study groups
print(f"\n  Study groups:")
for g in GROUP_ORDER:
    n = (adata.obs[STUDY_GROUP_COL] == g).sum()
    if n > 0:
        print(f"    {g}: {n:,}")

# Donors
print(f"\n  Donors: {adata.obs[SAMPLE_COL].nunique()}")

---
## 03 · Imbalance assessment

**Why.** Run before any CV is computed, because the imbalance decides whether
the comparison is interpretable at all.

Senescent cells are 2-5% of nuclei. Comparing the CV of a 400-cell group against
a 20,000-cell group is comparing two variance estimates with very different
precision, and the smaller group's CV is biased upward by the estimator itself
— not by biology.

**Display.** Cell counts per arm per donor, and a `balance_check` that states
plainly whether the arms can be compared or need subsampling.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# IMBALANCE ASSESSMENT
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("IMBALANCE ASSESSMENT")
print("=" * 60)

counts = adata.obs.groupby([SAMPLE_COL, 'is_senescent']).size().unstack(fill_value=0)
counts.columns = ['Non-SnC', 'SnC']
counts['total'] = counts.sum(axis=1)
counts['ratio'] = counts['Non-SnC'] / counts['SnC'].replace(0, np.nan)

print(f"\n  Per-donor summary:")
print(f"    Donors with both SnC & Non-SnC: {counts['SnC'].gt(0).sum()} / {len(counts)}")
print(f"    Median SnC per donor: {counts['SnC'].median():.0f}")
print(f"    Median Non-SnC per donor: {counts['Non-SnC'].median():.0f}")
print(f"    Median imbalance ratio: {counts['ratio'].median():.1f}x")

print(f"\n  {'Donor':<12} {'Non-SnC':>8} {'SnC':>6} {'Ratio':>8}")
print(f"  {'─'*38}")
for donor, row in counts.sort_values('ratio', ascending=False).iterrows():
    ratio_str = f"{row['ratio']:.1f}x" if not np.isnan(row['ratio']) else "inf"
    print(f"  {str(donor):<12} {row['Non-SnC']:>8,.0f} {row['SnC']:>6,.0f} {ratio_str:>8}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# BALANCED SUBSAMPLING
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("BALANCED SUBSAMPLING")
print("=" * 60)

MIN_CELLS_PER_GROUP = 10
np.random.seed(42)

keep_indices = []
skipped_donors = []

for donor, group_df in adata.obs.groupby(SAMPLE_COL):
    snc_idx = group_df[group_df['is_senescent'] == True].index
    nonsnc_idx = group_df[group_df['is_senescent'] == False].index

    n_min = min(len(snc_idx), len(nonsnc_idx))

    if n_min < MIN_CELLS_PER_GROUP:
        skipped_donors.append({'Donor': donor, 'n_snc': len(snc_idx), 'n_nonsnc': len(nonsnc_idx)})
        continue

    sampled_snc = np.random.choice(snc_idx, n_min, replace=False)
    sampled_nonsnc = np.random.choice(nonsnc_idx, n_min, replace=False)

    keep_indices.extend(sampled_snc)
    keep_indices.extend(sampled_nonsnc)

adata_balanced = adata[keep_indices].copy()

print(f"\n  Original:  {adata.n_obs:,} cells")
print(f"  Balanced:  {adata_balanced.n_obs:,} cells")
print(f"  Retained:  {100*adata_balanced.n_obs/adata.n_obs:.1f}%")

n_snc_bal = adata_balanced.obs['is_senescent'].sum()
n_nonsnc_bal = (~adata_balanced.obs['is_senescent']).sum()
print(f"\n  SnC:       {n_snc_bal:,}")
print(f"  Non-SnC:   {n_nonsnc_bal:,}")
print(f"  Balanced:  {'✓' if n_snc_bal == n_nonsnc_bal else '✗'}")

n_donors_kept = adata_balanced.obs[SAMPLE_COL].nunique()
print(f"\n  Donors kept:    {n_donors_kept}")
print(f"  Donors skipped: {len(skipped_donors)}")

if skipped_donors:
    print(f"\n  Skipped (< {MIN_CELLS_PER_GROUP} SnC):")
    for d in skipped_donors:
        print(f"    {d['Donor']}: {d['n_snc']} SnC, {d['n_nonsnc']} Non-SnC")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VERIFY BALANCE PER DONOR
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("BALANCE VERIFICATION")
print("=" * 60)

balance_check = (
    adata_balanced.obs.groupby([SAMPLE_COL, 'is_senescent'])
    .size().unstack(fill_value=0)
)
balance_check.columns = ['Non-SnC', 'SnC']
balance_check['equal'] = balance_check['Non-SnC'] == balance_check['SnC']

print(f"\n  {'Donor':<12} {'Non-SnC':>8} {'SnC':>6} {'Balanced':>8}")
print(f"  {'─'*38}")
for donor, row in balance_check.iterrows():
    check = "✓" if row['equal'] else "✗"
    print(f"  {str(donor):<12} {row['Non-SnC']:>8} {row['SnC']:>6} {check:>8}")

n_perfect = balance_check['equal'].sum()
print(f"\n  Perfectly balanced: {n_perfect} / {len(balance_check)} donors")
print(f"  Total SnC = {balance_check['SnC'].sum()}, Total Non-SnC = {balance_check['Non-SnC'].sum()}")

# Study group distribution in balanced set
print(f"\n  Study groups in balanced set:")
for g in GROUP_ORDER:
    n = (adata_balanced.obs[STUDY_GROUP_COL] == g).sum()
    nd = adata_balanced.obs[adata_balanced.obs[STUDY_GROUP_COL] == g][SAMPLE_COL].nunique()
    if n > 0:
        print(f"    {g}: {n:,} cells, {nd} donors")

In [ ]:
adata

In [ ]:
balance_check = (
    adata_balanced.obs.groupby([SAMPLE_COL, 'is_senescent'], observed=True)
    .size().unstack(fill_value=0)
)

# Set expression matrix to log-normalized
if 'lognorm' in adata_balanced.layers:
    adata_balanced.X = adata_balanced.layers['lognorm'].copy()
    print(f"  Set X = lognorm layer")
elif 'log_normalized' in adata_balanced.layers:
    adata_balanced.X = adata_balanced.layers['log_normalized'].copy()
    print(f"  Set X = log_normalized layer")
else:
    # Check if X is already log-normalized (range ~0-10, sparse)
    x_max = adata_balanced.X.max()
    if 4 < x_max < 15:
        print(f"  X already log-normalized (range 0–{x_max:.2f}), using as-is")
    else:
        print(f"  ⚠ X range 0–{x_max:.2f} — may need normalization")
        print(f"  Available layers: {list(adata_balanced.layers.keys())}")

---
## 04 · DEG results

**Why.** Loads the differential expression results so CV can be stratified by
whether a gene's mean moved. Without that split, a widened spread among DEGs is
ambiguous: a shifted mean drags the variance with it, so "more variable" and
"differentially expressed" would be partly the same statement.

**Display.** DEG counts and direction, as a summary figure.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# LOAD DEG RESULTS
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("LOAD DEG RESULTS")
print("=" * 60)

# List available comparisons
deg_files = list(DEG_RESULTS_DIR.glob("*.csv"))
deg_files = [f for f in deg_files if not any(x in f.name for x in ["summary", "gsea", "genes", "pseudobulk"])]

print("Available comparisons:")
for f in deg_files:
    print(f"  • {f.name}")

# Select comparison
COMPARISON = "SnCvsNonSnC_all"
deg_file = DEG_RESULTS_DIR / f"{DATASET}_{CELL_TYPE.lower()}_{COMPARISON}.csv"

if deg_file.exists():
    deg_data = pd.read_csv(deg_file)
    print(f"\n  Loaded: {COMPARISON}")
    print(f"  Total genes tested: {len(deg_data):,}")

    # Ensure direction column
    if "direction" not in deg_data.columns:
        deg_data["significant"] = (
            (deg_data["p_val_adj"] < FDR_THRESHOLD) &
            (deg_data["avg_log2FC"].abs() > LFC_THRESHOLD)
        )
        deg_data["direction"] = "NS"
        deg_data.loc[deg_data["significant"] & (deg_data["avg_log2FC"] > LFC_THRESHOLD), "direction"] = "Up"
        deg_data.loc[deg_data["significant"] & (deg_data["avg_log2FC"] < -LFC_THRESHOLD), "direction"] = "Down"

    n_up = (deg_data["direction"] == "Up").sum()
    n_down = (deg_data["direction"] == "Down").sum()

    print(f"\n  DEG Summary:")
    print(f"    Up in SnC:   {n_up}")
    print(f"    Down in SnC: {n_down}")
    print(f"    Total DEGs:  {n_up + n_down}")

    HAS_DEG = True
else:
    print(f"\n  ⚠ DEG file not found: {deg_file.name}")
    print(f"  Skipping DEG-dependent analyses")
    HAS_DEG = False

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# DEG BAR PLOT + VOLCANO
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("DEG VISUALIZATION")
print("=" * 60)

if HAS_DEG:
    try:
        from adjustText import adjust_text
    except ImportError:
        import subprocess
        subprocess.check_call(["pip", "install", "adjustText", "-q"])
        from adjustText import adjust_text

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))

    # ── Panel A: Bar plot ──
    ax = axes[0]
    ax.grid(False)
    bars = ax.bar(['Up', 'Down'], [n_up, n_down], color=['#D73027', '#4575B4'],
                  edgecolor='none', width=0.5)
    for bar, count in zip(bars, [n_up, n_down]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(count), ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_ylabel('Number of DEGs', fontsize=9)
    ax.set_title('A. DEG Counts (SnC vs Non-SnC)', fontsize=10, fontweight='bold', loc='left')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.tick_params(labelsize=8)

    # ── Panel B: Volcano ──
    ax = axes[1]
    ax.grid(False)

    deg_plot = deg_data.copy()
    deg_plot['-log10p'] = -np.log10(deg_plot['p_val_adj'].clip(lower=1e-300))

    # NS points
    ns = deg_plot[deg_plot['direction'] == 'NS']
    ax.scatter(ns['avg_log2FC'], ns['-log10p'], s=2, c='#CCCCCC', alpha=0.3, rasterized=True)

    # Up
    up = deg_plot[deg_plot['direction'] == 'Up']
    ax.scatter(up['avg_log2FC'], up['-log10p'], s=8, c='#D73027', alpha=0.6, label=f'Up ({n_up})')

    # Down
    down = deg_plot[deg_plot['direction'] == 'Down']
    ax.scatter(down['avg_log2FC'], down['-log10p'], s=8, c='#4575B4', alpha=0.6, label=f'Down ({n_down})')

    # Thresholds
    ax.axhline(-np.log10(FDR_THRESHOLD), color='k', linestyle='--', lw=1, alpha=0.5)
    ax.axvline(LFC_THRESHOLD, color='k', linestyle='--', lw=0.5, alpha=0.5)
    ax.axvline(-LFC_THRESHOLD, color='k', linestyle='--', lw=0.5, alpha=0.5)

    # Label top genes
    if 'gene' in deg_plot.columns:
        gene_col = 'gene'
    elif deg_plot.index.name:
        deg_plot['gene'] = deg_plot.index
        gene_col = 'gene'
    else:
        gene_col = None

    if gene_col:
        top_genes = pd.concat([
            up.nlargest(5, '-log10p'),
            down.nlargest(3, '-log10p'),
        ])
        texts = []
        for _, row in top_genes.iterrows():
            texts.append(ax.text(row['avg_log2FC'], row['-log10p'], row[gene_col],
                                 fontsize=5.5, alpha=0.8))
        if texts:
            adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', lw=0.3))

    ax.set_xlabel('log₂FC (SnC / Non-SnC)', fontsize=9)
    ax.set_ylabel('-log₁₀(FDR)', fontsize=9)
    ax.set_title('B. Volcano Plot', fontsize=10, fontweight='bold', loc='left')
    ax.legend(fontsize=7, frameon=False, loc='upper left')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.tick_params(labelsize=7)

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_deg_summary.png', dpi=300, bbox_inches='tight')
    plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_deg_summary.svg', dpi=300, bbox_inches='tight')
    plt.show()
    print("Saved DEG summary figure")
else:
    print("  Skipped — no DEG data")

---
## 05 · Donor-level depth by senescence

**Why.** The second half of the confound check. Senescent cells carry more UMIs,
and UMI count sets the floor on sampling noise — more reads means less Poisson
noise per cell, which *lowers* CV independently of biology.

This section measures the depth difference per donor so the direction of the
bias is known before the CV comparison is read. It works against the group-size
bias in section 03, so the two do not simply add.

**Display.** Per-donor mean UMI, senescent versus non-senescent.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# DONOR-LEVEL UMI STATS
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("DONOR-LEVEL UMI STATS")
print("=" * 60)

from scipy.stats import wilcoxon

donor_stats = []

for donor in adata_balanced.obs[SAMPLE_COL].unique():
    donor_mask = adata_balanced.obs[SAMPLE_COL] == donor
    study_group = adata_balanced.obs.loc[donor_mask, STUDY_GROUP_COL].iloc[0]

    for is_snc in [True, False]:
        cell_mask = donor_mask & (adata_balanced.obs['is_senescent'] == is_snc)
        n_cells = cell_mask.sum()
        if n_cells == 0:
            continue

        if 'total_counts' in adata_balanced.obs.columns:
            median_umi = adata_balanced.obs.loc[cell_mask, 'total_counts'].median()
            mean_umi = adata_balanced.obs.loc[cell_mask, 'total_counts'].mean()
        else:
            median_umi = np.nan
            mean_umi = np.nan

        donor_stats.append({
            'donor': donor,
            'Study_Group': study_group,
            'Senescence': 'SnC' if is_snc else 'Non-SnC',
            'n_cells': n_cells,
            'median_UMI': median_umi,
            'mean_UMI': mean_umi,
        })

donor_stats = pd.DataFrame(donor_stats)

pivot_umi = donor_stats.pivot(index='donor', columns='Senescence', values='median_UMI').dropna()
stat_umi, pval_umi = wilcoxon(pivot_umi['SnC'], pivot_umi['Non-SnC'])

print(f"\n  Median UMI (SnC):     {pivot_umi['SnC'].median():,.0f}")
print(f"  Median UMI (Non-SnC): {pivot_umi['Non-SnC'].median():,.0f}")
print(f"  Wilcoxon paired: p = {pval_umi:.2e}")

# Figure
fig, ax = plt.subplots(figsize=(3, 3.5))
ax.grid(False)

sns.violinplot(data=donor_stats, x='Senescence', y='median_UMI',
               order=['Non-SnC', 'SnC'], palette=SNC_COLORS,
               inner=None, linewidth=0.5, alpha=0.6, ax=ax)
sns.stripplot(data=donor_stats, x='Senescence', y='median_UMI',
              order=['Non-SnC', 'SnC'], palette=SNC_COLORS,
              size=3, alpha=0.6, jitter=0.15, ax=ax)

ax.set_ylabel('Median UMI per donor', fontsize=9)
ax.set_xlabel('')
ax.set_title(f'{CELL_TYPE}: UMI by Senescence', fontsize=10, fontweight='bold')
ax.text(0.03, 0.97, f'p = {pval_umi:.1e}', transform=ax.transAxes,
        fontsize=7, ha='left', va='top', color='#555')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_umi_violin.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_umi_violin.svg', dpi=300, bbox_inches='tight')
plt.show()
print("Saved UMI violin figure")

---
## 06 · Highly variable genes on the balanced data

**Why.** HVG selection is run on the **balanced** subsample, not on the full
object. Selecting on the unbalanced data would let the larger arm determine
which genes are called variable, and those genes would then be the ones the CV
comparison runs on — a selection effect pointing at the answer.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# HVG SELECTION ON BALANCED DATA
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("HVG SELECTION")
print("=" * 60)

# Set expression matrix to log-normalized
if 'log_normalized' in adata_balanced.layers:
    adata_balanced.X = adata_balanced.layers['log_normalized'].copy()
    print("  Set X = log_normalized")
elif 'lognorm' in adata_balanced.layers:
    adata_balanced.X = adata_balanced.layers['lognorm'].copy()
    print("  Set X = lognorm")
else:
    print(f"  Using existing X | Layers: {list(adata_balanced.layers.keys())}")

# Compute HVGs on balanced data
N_TOP_GENES = 2000

sc.pp.highly_variable_genes(adata_balanced, n_top_genes=N_TOP_GENES, flavor='seurat_v3',
                            layer='raw_counts' if 'raw_counts' in adata_balanced.layers else None)

n_hvg = adata_balanced.var['highly_variable'].sum()
n_nonhvg = (~adata_balanced.var['highly_variable']).sum()

print(f"\n  Total genes: {adata_balanced.n_vars:,}")
print(f"  HVGs: {n_hvg:,}")
print(f"  Non-HVGs: {n_nonhvg:,}")

# Store gene status
adata_balanced.var['gene_status'] = 'Non-HVG'
adata_balanced.var.loc[adata_balanced.var['highly_variable'], 'gene_status'] = 'HVG'

print(f"\n  Top 10 HVGs by variance:")
hvg_info = adata_balanced.var[adata_balanced.var['highly_variable']].copy()
if 'highly_variable_rank' in hvg_info.columns:
    for gene in hvg_info.sort_values('highly_variable_rank').head(10).index:
        print(f"    {gene}")
else:
    print(f"    {list(hvg_info.head(10).index)}")

---
## 07 · Gene-level CV, stratified by DEG status

**Why.** The main test. Coefficient of variation per gene, senescent versus
non-senescent, split by whether that gene is a DEG.

**Formula.** `CV = sd(expression) / mean(expression)` per gene per arm.

The non-DEG stratum is the informative one: those genes have no detected mean
shift, so a CV difference there is variability changing on its own rather than
as a consequence of the mean moving.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# GENE-LEVEL CV: STRATIFIED BY DEG STATUS
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("GENE-LEVEL CV (DEG-STRATIFIED)")
print("=" * 60)

# Set expression matrix
if 'log_normalized' in adata_balanced.layers:
    adata_balanced.X = adata_balanced.layers['log_normalized'].copy()
    print("  Set X = log_normalized")
elif 'lognorm' in adata_balanced.layers:
    adata_balanced.X = adata_balanced.layers['lognorm'].copy()
    print("  Set X = lognorm")
else:
    print(f"  Using existing X | Layers: {list(adata_balanced.layers.keys())}")

# Split by senescence
snc_mask = adata_balanced.obs['is_senescent'].values
nonsnc_mask = ~snc_mask

X_snc = adata_balanced.X[snc_mask]
X_nonsnc = adata_balanced.X[nonsnc_mask]

if issparse(X_snc):
    X_snc = X_snc.toarray()
    X_nonsnc = X_nonsnc.toarray()

print(f"\n  SnC cells:     {X_snc.shape[0]:,}")
print(f"  Non-SnC cells: {X_nonsnc.shape[0]:,}")
print(f"  Genes:         {X_snc.shape[1]:,}")

# Compute CV per gene
mean_snc = X_snc.mean(axis=0)
std_snc = X_snc.std(axis=0)
mean_nonsnc = X_nonsnc.mean(axis=0)
std_nonsnc = X_nonsnc.std(axis=0)

MIN_MEAN = 0.01
valid_mask = (mean_snc > MIN_MEAN) & (mean_nonsnc > MIN_MEAN)

cv_data = pd.DataFrame({
    'gene': adata_balanced.var_names,
    'mean_SnC': mean_snc,
    'mean_NonSnC': mean_nonsnc,
    'cv_SnC': np.where(valid_mask, std_snc / mean_snc, np.nan),
    'cv_NonSnC': np.where(valid_mask, std_nonsnc / mean_nonsnc, np.nan),
}).dropna(subset=['cv_SnC', 'cv_NonSnC'])

cv_data['cv_diff'] = cv_data['cv_SnC'] - cv_data['cv_NonSnC']
cv_data['cv_ratio'] = cv_data['cv_SnC'] / cv_data['cv_NonSnC']

print(f"  Genes with mean > {MIN_MEAN}: {len(cv_data):,}")

# ── Map DEG status onto CV data ──
if HAS_DEG:
    # Identify gene column in deg_data
    if 'gene' in deg_data.columns:
        deg_gene_col = 'gene'
    else:
        deg_data['gene'] = deg_data.index
        deg_gene_col = 'gene'

    deg_map = deg_data.set_index(deg_gene_col)['direction'].to_dict()
    cv_data['DEG_status'] = cv_data['gene'].map(deg_map).fillna('Not_Tested')

    # Genes not in DEG results but passing mean filter → treat as NS
    cv_data.loc[cv_data['DEG_status'] == 'Not_Tested', 'DEG_status'] = 'NS'

    print(f"\n  DEG status mapped:")
    for status in ['Up', 'Down', 'NS']:
        n = (cv_data['DEG_status'] == status).sum()
        print(f"    {status}: {n:,} genes")
else:
    cv_data['DEG_status'] = 'All'
    print("  No DEG data — treating all genes as one group")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CV STATISTICS BY DEG CATEGORY
# ════════════════════════════════════════════════════════════════════════════════

from scipy.stats import wilcoxon

print("=" * 60)
print("CV STATISTICS BY DEG CATEGORY")
print("=" * 60)

cv_stats = []

for status in ['Up', 'Down', 'NS']:
    subset = cv_data[cv_data['DEG_status'] == status]
    if len(subset) < 10:
        print(f"  {status}: skipped (n={len(subset)})")
        continue

    stat, pval = wilcoxon(subset['cv_SnC'], subset['cv_NonSnC'])
    med_snc = subset['cv_SnC'].median()
    med_nonsnc = subset['cv_NonSnC'].median()
    med_diff = subset['cv_diff'].median()
    n_higher_snc = (subset['cv_diff'] > 0).sum()
    pct_higher = 100 * n_higher_snc / len(subset)
    direction = "SnC > Non-SnC" if med_diff > 0 else "Non-SnC > SnC"

    cv_stats.append({
        'DEG_status': status,
        'n_genes': len(subset),
        'median_cv_SnC': med_snc,
        'median_cv_NonSnC': med_nonsnc,
        'median_cv_diff': med_diff,
        'pct_higher_SnC': pct_higher,
        'wilcoxon_p': pval,
        'direction': direction,
    })

    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
    print(f"\n  {status} DEGs (n={len(subset):,}):")
    print(f"    Median CV SnC:     {med_snc:.4f}")
    print(f"    Median CV Non-SnC: {med_nonsnc:.4f}")
    print(f"    ΔCV (SnC-NonSnC):  {med_diff:.4f}")
    print(f"    Higher in SnC:     {n_higher_snc:,} / {len(subset):,} ({pct_higher:.1f}%)")
    print(f"    Wilcoxon p:        {pval:.2e} {sig}")
    print(f"    Direction:         {direction}")

# Also run on ALL genes
stat_all, pval_all = wilcoxon(cv_data['cv_SnC'], cv_data['cv_NonSnC'])
med_all_diff = cv_data['cv_diff'].median()
cv_stats.append({
    'DEG_status': 'All',
    'n_genes': len(cv_data),
    'median_cv_SnC': cv_data['cv_SnC'].median(),
    'median_cv_NonSnC': cv_data['cv_NonSnC'].median(),
    'median_cv_diff': med_all_diff,
    'pct_higher_SnC': 100 * (cv_data['cv_diff'] > 0).sum() / len(cv_data),
    'wilcoxon_p': pval_all,
    'direction': "SnC > Non-SnC" if med_all_diff > 0 else "Non-SnC > SnC",
})

print(f"\n  ALL genes (n={len(cv_data):,}):")
print(f"    ΔCV: {med_all_diff:.4f}, p={pval_all:.2e}")

df_cv_stats = pd.DataFrame(cv_stats)

---
## 08 · CV visualization, DEG-stratified

**Why.** Distributions rather than summary statistics. A median CV shift can
come from a uniform move or from a tail — and those mean different things.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CV VISUALIZATION: STRATIFIED + PER STUDY GROUP
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("CV VISUALIZATION (DEG-STRATIFIED)")
print("=" * 60)

from scipy.stats import wilcoxon, mannwhitneyu
from matplotlib.patches import Patch

DEG_COLORS = {'Up': '#D73027', 'Down': '#4575B4', 'NS': '#CCCCCC', 'All': '#808080'}

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

# ── Panel A: Scatter ──
ax = axes[0, 0]
ax.grid(False)

for status, color in [('NS', '#CCCCCC'), ('Down', '#4575B4'), ('Up', '#D73027')]:
    sub = cv_data[cv_data['DEG_status'] == status]
    alpha = 0.15 if status == 'NS' else 0.6
    size = 2 if status == 'NS' else 6
    ax.scatter(sub['cv_NonSnC'], sub['cv_SnC'], s=size, c=color, alpha=alpha,
               label=f'{status} ({len(sub):,})', rasterized=True)

lim_max = max(cv_data['cv_SnC'].quantile(0.99), cv_data['cv_NonSnC'].quantile(0.99))
ax.plot([0, lim_max], [0, lim_max], 'k--', lw=0.8, alpha=0.5)
ax.set_xlabel('CV (Non-SnC)', fontsize=9); ax.set_ylabel('CV (SnC)', fontsize=9)
ax.set_title('A. Gene-level CV by DEG Status', fontsize=10, fontweight='bold', loc='left')
ax.legend(fontsize=7, frameon=False, loc='upper left')
ax.set_xlim(0, lim_max); ax.set_ylim(0, lim_max)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=7)

# ── Panel B: ΔCV violin by DEG status + stats ──
ax = axes[0, 1]
ax.grid(False)

statuses_b = ['Up', 'Down', 'NS']
vp_data_b = [cv_data[cv_data['DEG_status'] == s]['cv_diff'].values for s in statuses_b]

parts_b = ax.violinplot(vp_data_b, positions=[0, 1, 2], showextrema=False, showmedians=True)
for i, (body, status) in enumerate(zip(parts_b['bodies'], statuses_b)):
    body.set_facecolor(DEG_COLORS[status]); body.set_alpha(0.7)
parts_b['cmedians'].set_color('black'); parts_b['cmedians'].set_linewidth(1.5)

ax.axhline(0, color='k', linestyle='--', lw=0.8, alpha=0.5)

# Stats: test if ΔCV != 0 per category
for i, status in enumerate(statuses_b):
    vals = cv_data[cv_data['DEG_status'] == status]['cv_diff'].values
    if len(vals) > 10:
        _, p = wilcoxon(vals)
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        y_top = np.percentile(vals, 97)
        ax.text(i, y_top * 1.05, sig, ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xticks([0, 1, 2]); ax.set_xticklabels(['Up', 'Down', 'NS'], fontsize=8)
ax.set_ylabel('ΔCV (SnC − Non-SnC)', fontsize=9)
ax.set_title('B. CV Difference by DEG Status', fontsize=10, fontweight='bold', loc='left')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=7)

# ── Panel C: Violin CV by DEG status × senescence + stats ──
ax = axes[0, 2]
ax.grid(False)

positions_c = []
vp_data_c = []
vp_colors_c = []

for i, status in enumerate(['Up', 'Down', 'NS']):
    sub = cv_data[cv_data['DEG_status'] == status]
    for j, (sen, col) in enumerate([('Non-SnC', 'cv_NonSnC'), ('SnC', 'cv_SnC')]):
        vals = sub[col].values
        if len(vals) > 0:
            positions_c.append(i * 3 + j * 1.0)
            vp_data_c.append(vals)
            vp_colors_c.append(SNC_COLORS[sen])

parts_c = ax.violinplot(vp_data_c, positions=positions_c, showextrema=False, showmedians=True, widths=0.8)
for body, color in zip(parts_c['bodies'], vp_colors_c):
    body.set_facecolor(color); body.set_alpha(0.7)
parts_c['cmedians'].set_color('black'); parts_c['cmedians'].set_linewidth(1.2)

# Stats per DEG category (paired by gene)
for i, status in enumerate(['Up', 'Down', 'NS']):
    sub = cv_data[cv_data['DEG_status'] == status]
    if len(sub) > 10:
        _, p = wilcoxon(sub['cv_SnC'], sub['cv_NonSnC'])
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        y_top = max(sub['cv_SnC'].quantile(0.97), sub['cv_NonSnC'].quantile(0.97))
        ax.text(i * 3 + 0.5, y_top * 1.05, sig, ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xticks([i * 3 + 0.5 for i in range(3)])
ax.set_xticklabels(['Up', 'Down', 'NS'], fontsize=8)
ax.set_ylabel('CV', fontsize=9)
ax.set_title('C. CV by DEG Status × Senescence', fontsize=10, fontweight='bold', loc='left')
ax.legend(handles=[Patch(facecolor=SNC_COLORS['Non-SnC'], label='Non-SnC'),
                   Patch(facecolor=SNC_COLORS['SnC'], label='SnC')],
          fontsize=7, frameon=False, loc='upper right')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=7)

# ── Panel D: % genes with higher CV in SnC ──
ax = axes[1, 0]
ax.grid(False)

bar_data = df_cv_stats[df_cv_stats['DEG_status'].isin(['Up', 'Down', 'NS', 'All'])].copy()
bar_colors = [DEG_COLORS.get(s, '#808080') for s in bar_data['DEG_status']]

bars = ax.bar(range(len(bar_data)), bar_data['pct_higher_SnC'], color=bar_colors, edgecolor='none')
ax.axhline(50, color='k', linestyle='--', lw=0.8, alpha=0.5)

for i, (_, row) in enumerate(bar_data.iterrows()):
    sig = '***' if row['wilcoxon_p'] < 0.001 else '**' if row['wilcoxon_p'] < 0.01 else '*' if row['wilcoxon_p'] < 0.05 else 'ns'
    ax.text(i, row['pct_higher_SnC'] + 1, sig, ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(range(len(bar_data)))
ax.set_xticklabels(bar_data['DEG_status'], fontsize=8)
ax.set_ylabel('% Genes with Higher CV in SnC', fontsize=9)
ax.set_ylim(0, 100)
ax.set_title('D. Direction of CV Difference', fontsize=10, fontweight='bold', loc='left')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=7)

# ── Panel E: Violin CV by Study Group (Up DEGs) + stats ──
ax = axes[1, 1]
ax.grid(False)

study_groups = [g for g in GROUP_ORDER if g in adata_balanced.obs[STUDY_GROUP_COL].values]
up_genes = cv_data[cv_data['DEG_status'] == 'Up']['gene'].values
up_idx = [list(adata_balanced.var_names).index(g) for g in up_genes if g in adata_balanced.var_names]

sg_cv_up = []
for sg in study_groups:
    sg_mask = adata_balanced.obs[STUDY_GROUP_COL] == sg
    for is_snc in [True, False]:
        cell_mask = sg_mask & (adata_balanced.obs['is_senescent'] == is_snc)
        if cell_mask.sum() < 20:
            continue
        X_sub = adata_balanced.X[cell_mask.values][:, up_idx]
        if issparse(X_sub): X_sub = X_sub.toarray()
        means = X_sub.mean(axis=0)
        stds = X_sub.std(axis=0)
        valid = means > MIN_MEAN
        if valid.sum() < 5: continue
        cvs = stds[valid] / means[valid]
        for cv_val in cvs:
            sg_cv_up.append({'Study_Group': sg, 'Senescence': 'SnC' if is_snc else 'Non-SnC', 'CV': float(cv_val)})

sg_cv_up_df = pd.DataFrame(sg_cv_up)

positions_e = []
vp_data_e = []
vp_colors_e = []

for i, sg in enumerate(study_groups):
    for j, sen in enumerate(['Non-SnC', 'SnC']):
        vals = sg_cv_up_df[(sg_cv_up_df['Study_Group'] == sg) & (sg_cv_up_df['Senescence'] == sen)]['CV'].values
        if len(vals) > 5:
            positions_e.append(i * 2.5 + j * 0.8)
            vp_data_e.append(vals)
            vp_colors_e.append(SNC_COLORS[sen])

if vp_data_e:
    parts_e = ax.violinplot(vp_data_e, positions=positions_e, showextrema=False, showmedians=True, widths=0.6)
    for body, color in zip(parts_e['bodies'], vp_colors_e):
        body.set_facecolor(color); body.set_alpha(0.7)
    parts_e['cmedians'].set_color('black'); parts_e['cmedians'].set_linewidth(1.0)

# Stats per study group
for i, sg in enumerate(study_groups):
    snc_vals = sg_cv_up_df[(sg_cv_up_df['Study_Group'] == sg) & (sg_cv_up_df['Senescence'] == 'SnC')]['CV'].values
    nonsnc_vals = sg_cv_up_df[(sg_cv_up_df['Study_Group'] == sg) & (sg_cv_up_df['Senescence'] == 'Non-SnC')]['CV'].values
    if len(snc_vals) > 5 and len(nonsnc_vals) > 5:
        _, p = mannwhitneyu(snc_vals, nonsnc_vals, alternative='two-sided')
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        y_top = max(np.percentile(snc_vals, 95), np.percentile(nonsnc_vals, 95))
        ax.text(i * 2.5 + 0.4, y_top * 1.05, sig, ha='center', va='bottom', fontsize=6, fontweight='bold')

ax.set_xticks([i * 2.5 + 0.4 for i in range(len(study_groups))])
ax.set_xticklabels([g.replace('_', '\n').replace('Age\n', '') for g in study_groups],
                   fontsize=6, rotation=45, ha='right')
ax.set_ylabel('CV (Up DEGs)', fontsize=9)
ax.set_title('E. CV by Study Group (Up DEGs)', fontsize=10, fontweight='bold', loc='left')
ax.legend(handles=[Patch(facecolor=SNC_COLORS['Non-SnC'], label='Non-SnC'),
                   Patch(facecolor=SNC_COLORS['SnC'], label='SnC')],
          fontsize=6, frameon=False, loc='upper right')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=7)

# ── Panel F: Violin CV by Study Group (Down DEGs) + stats ──
ax = axes[1, 2]
ax.grid(False)

down_genes = cv_data[cv_data['DEG_status'] == 'Down']['gene'].values
down_idx = [list(adata_balanced.var_names).index(g) for g in down_genes if g in adata_balanced.var_names]

sg_cv_down = []
for sg in study_groups:
    sg_mask = adata_balanced.obs[STUDY_GROUP_COL] == sg
    for is_snc in [True, False]:
        cell_mask = sg_mask & (adata_balanced.obs['is_senescent'] == is_snc)
        if cell_mask.sum() < 20:
            continue
        X_sub = adata_balanced.X[cell_mask.values][:, down_idx]
        if issparse(X_sub): X_sub = X_sub.toarray()
        means = X_sub.mean(axis=0)
        stds = X_sub.std(axis=0)
        valid = means > MIN_MEAN
        if valid.sum() < 5: continue
        cvs = stds[valid] / means[valid]
        for cv_val in cvs:
            sg_cv_down.append({'Study_Group': sg, 'Senescence': 'SnC' if is_snc else 'Non-SnC', 'CV': float(cv_val)})

sg_cv_down_df = pd.DataFrame(sg_cv_down)

positions_f = []
vp_data_f = []
vp_colors_f = []

for i, sg in enumerate(study_groups):
    for j, sen in enumerate(['Non-SnC', 'SnC']):
        vals = sg_cv_down_df[(sg_cv_down_df['Study_Group'] == sg) & (sg_cv_down_df['Senescence'] == sen)]['CV'].values
        if len(vals) > 5:
            positions_f.append(i * 2.5 + j * 0.8)
            vp_data_f.append(vals)
            vp_colors_f.append(SNC_COLORS[sen])

if vp_data_f:
    parts_f = ax.violinplot(vp_data_f, positions=positions_f, showextrema=False, showmedians=True, widths=0.6)
    for body, color in zip(parts_f['bodies'], vp_colors_f):
        body.set_facecolor(color); body.set_alpha(0.7)
    parts_f['cmedians'].set_color('black'); parts_f['cmedians'].set_linewidth(1.0)

# Stats per study group
for i, sg in enumerate(study_groups):
    snc_vals = sg_cv_down_df[(sg_cv_down_df['Study_Group'] == sg) & (sg_cv_down_df['Senescence'] == 'SnC')]['CV'].values
    nonsnc_vals = sg_cv_down_df[(sg_cv_down_df['Study_Group'] == sg) & (sg_cv_down_df['Senescence'] == 'Non-SnC')]['CV'].values
    if len(snc_vals) > 5 and len(nonsnc_vals) > 5:
        _, p = mannwhitneyu(snc_vals, nonsnc_vals, alternative='two-sided')
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        y_top = max(np.percentile(snc_vals, 95), np.percentile(nonsnc_vals, 95))
        ax.text(i * 2.5 + 0.4, y_top * 1.05, sig, ha='center', va='bottom', fontsize=6, fontweight='bold')

ax.set_xticks([i * 2.5 + 0.4 for i in range(len(study_groups))])
ax.set_xticklabels([g.replace('_', '\n').replace('Age\n', '') for g in study_groups],
                   fontsize=6, rotation=45, ha='right')
ax.set_ylabel('CV (Down DEGs)', fontsize=9)
ax.set_title('F. CV by Study Group (Down DEGs)', fontsize=10, fontweight='bold', loc='left')
ax.legend(handles=[Patch(facecolor=SNC_COLORS['Non-SnC'], label='Non-SnC'),
                   Patch(facecolor=SNC_COLORS['SnC'], label='SnC')],
          fontsize=6, frameon=False, loc='upper right')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_deg_stratified.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_deg_stratified.svg', dpi=300, bbox_inches='tight')
plt.show()
print("Saved DEG-stratified CV figure (6 panels + stats)")

---
## 09 · Donor-level CV, DEG-stratified

**Why.** Section 07 pools cells across donors, which pushes between-donor
variance into the CV. Recomputing per donor and then comparing donor-level
values keeps the unit of analysis where it is everywhere else in this pipeline.

If the pooled and donor-level results disagree, the pooled CV was measuring
donor heterogeneity, not cell-to-cell heterogeneity.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# DONOR-LEVEL CV: DEG-STRATIFIED
# ════════════════════════════════════════════════════════════════════════════════

from scipy.stats import wilcoxon

print("=" * 60)
print("DONOR-LEVEL CV (DEG-STRATIFIED)")
print("=" * 60)

# Get gene indices per DEG category
deg_gene_sets = {}
for status in ['Up', 'Down', 'NS']:
    genes = cv_data[cv_data['DEG_status'] == status]['gene'].values
    gene_idx = [list(adata_balanced.var_names).index(g) for g in genes if g in adata_balanced.var_names]
    deg_gene_sets[status] = gene_idx
    print(f"  {status}: {len(gene_idx)} genes in expression matrix")

donor_cv_deg = []

for donor in adata_balanced.obs[SAMPLE_COL].unique():
    donor_mask = adata_balanced.obs[SAMPLE_COL] == donor
    study_group = adata_balanced.obs.loc[donor_mask, STUDY_GROUP_COL].iloc[0]

    for is_snc in [True, False]:
        cell_mask = donor_mask & (adata_balanced.obs['is_senescent'] == is_snc)
        n_cells = cell_mask.sum()

        if n_cells < MIN_CELLS_PER_GROUP:
            continue

        X_sub = adata_balanced.X[cell_mask.values]
        if issparse(X_sub):
            X_sub = X_sub.toarray()

        for status, gene_idx in deg_gene_sets.items():
            if len(gene_idx) < 10:
                continue

            X_genes = X_sub[:, gene_idx]
            means = X_genes.mean(axis=0)
            stds = X_genes.std(axis=0)
            valid = means > MIN_MEAN
            if valid.sum() < 5:
                continue

            cvs = stds[valid] / means[valid]

            donor_cv_deg.append({
                'donor': donor,
                'Study_Group': study_group,
                'Senescence': 'SnC' if is_snc else 'Non-SnC',
                'DEG_status': status,
                'n_cells': n_cells,
                'median_cv': float(np.median(cvs)),
                'mean_cv': float(cvs.mean()),
                'n_genes': int(valid.sum()),
            })

donor_cv_deg = pd.DataFrame(donor_cv_deg)
print(f"\n  Total observations: {len(donor_cv_deg)}")

# Paired tests per DEG category
print(f"\n  {'DEG Status':<10} {'n_paired':>8} {'SnC CV':>10} {'NonSnC CV':>10} {'p':>10} {'Direction':<15}")
print(f"  {'─'*65}")

donor_cv_paired_stats = []

for status in ['Up', 'Down', 'NS']:
    sub = donor_cv_deg[donor_cv_deg['DEG_status'] == status]
    donors_s = set(sub[sub['Senescence'] == 'SnC']['donor'])
    donors_n = set(sub[sub['Senescence'] == 'Non-SnC']['donor'])
    paired = sorted(donors_s & donors_n)

    if len(paired) < 5:
        print(f"  {status:<10} {'<5 paired':>8}")
        continue

    pivot = sub[sub['donor'].isin(paired)].pivot(index='donor', columns='Senescence', values='median_cv')
    stat, pval = wilcoxon(pivot['SnC'], pivot['Non-SnC'])
    med_s = pivot['SnC'].median()
    med_n = pivot['Non-SnC'].median()
    direction = "SnC > Non-SnC" if med_s > med_n else "Non-SnC > SnC"

    donor_cv_paired_stats.append({
        'DEG_status': status, 'n_paired': len(paired),
        'median_cv_SnC': med_s, 'median_cv_NonSnC': med_n,
        'wilcoxon_p': pval, 'direction': direction,
    })

    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
    print(f"  {status:<10} {len(paired):>8} {med_s:>10.4f} {med_n:>10.4f} {pval:>10.2e} {direction:<15} {sig}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# DONOR-LEVEL CV FIGURE (DEG-STRATIFIED)
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("DONOR-LEVEL CV FIGURE")
print("=" * 60)

import matplotlib.cm as cm

statuses = ['Up', 'Down', 'NS']

fig, axes = plt.subplots(1, 3, figsize=(7, 2.8))

for ax, status in zip(axes, statuses):
    ax.grid(False)
    sub = donor_cv_deg[donor_cv_deg['DEG_status'] == status]

    donors_s = set(sub[sub['Senescence'] == 'SnC']['donor'])
    donors_n = set(sub[sub['Senescence'] == 'Non-SnC']['donor'])
    paired = sorted(donors_s & donors_n)

    if len(paired) < 5:
        ax.text(0.5, 0.5, f'{status}\nn < 5', ha='center', va='center',
                transform=ax.transAxes, fontsize=8, color='gray')
        for spine in ax.spines.values(): spine.set_visible(False)
        ax.set_xticks([]); ax.set_yticks([])
        continue

    pivot = sub[sub['donor'].isin(paired)].pivot(index='donor', columns='Senescence', values='median_cv')

    # Unique color per donor
    donor_cmap = cm.get_cmap('tab20', len(paired))
    donor_colors = {d: donor_cmap(i) for i, d in enumerate(paired)}

    # Slim boxplots
    bp = ax.boxplot([pivot['Non-SnC'], pivot['SnC']],
                    positions=[0, 0.6], widths=0.2, showfliers=False, patch_artist=True,
                    boxprops=dict(facecolor='white', edgecolor='#999', linewidth=0.6),
                    medianprops=dict(color='k', linewidth=1.2),
                    whiskerprops=dict(color='#999', linewidth=0.5),
                    capprops=dict(color='#999', linewidth=0.5))

    # Paired lines + jittered dots, colored per donor
    for donor in pivot.index:
        c = donor_colors[donor]
        jx = np.random.uniform(-0.04, 0.04)
        ax.plot([0 + jx, 0.6 + jx],
                [pivot.loc[donor, 'Non-SnC'], pivot.loc[donor, 'SnC']],
                c=c, alpha=0.4, lw=0.6, zorder=1)
        ax.scatter(0 + jx, pivot.loc[donor, 'Non-SnC'],
                   c=[c], s=10, edgecolor='white', linewidth=0.2, zorder=3)
        ax.scatter(0.6 + jx, pivot.loc[donor, 'SnC'],
                   c=[c], s=10, edgecolor='white', linewidth=0.2, zorder=3)

    # Stats
    stat_row = [s for s in donor_cv_paired_stats if s['DEG_status'] == status]
    if stat_row:
        pval = stat_row[0]['wilcoxon_p']
        sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
        ax.text(0.03, 0.97, f'p={pval:.1e} {sig}\nn={len(paired)}',
                transform=ax.transAxes, fontsize=5.5, ha='left', va='top', color='#555')

    ax.set_xlim(-0.25, 0.85)
    ax.set_xticks([0, 0.6])
    ax.set_xticklabels(['Non-SnC', 'SnC'], fontsize=7)
    ax.set_title(f'{status} DEGs', fontsize=8, fontweight='bold',
                 color=DEG_COLORS.get(status, '#333'), pad=3)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.5); ax.spines['left'].set_linewidth(0.5)
    ax.tick_params(axis='y', labelsize=6)

    if ax == axes[0]:
        ax.set_ylabel('Median CV', fontsize=8)

plt.subplots_adjust(wspace=0.35)
plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_donor_deg.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_donor_deg.svg', dpi=300, bbox_inches='tight')
plt.show()

# ════════════════════════════════════════════════════════════════════════════════
# DONOR-LEVEL CV FIGURE — CLEAN (NO DOTS)
# ════════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(5.5, 2.5))

for ax, status in zip(axes, statuses):
    ax.grid(False)
    sub = donor_cv_deg[donor_cv_deg['DEG_status'] == status]

    donors_s = set(sub[sub['Senescence'] == 'SnC']['donor'])
    donors_n = set(sub[sub['Senescence'] == 'Non-SnC']['donor'])
    paired = sorted(donors_s & donors_n)

    if len(paired) < 5:
        ax.text(0.5, 0.5, f'{status}\nn < 5', ha='center', va='center',
                transform=ax.transAxes, fontsize=8, color='gray')
        for spine in ax.spines.values(): spine.set_visible(False)
        ax.set_xticks([]); ax.set_yticks([])
        continue

    pivot = sub[sub['donor'].isin(paired)].pivot(index='donor', columns='Senescence', values='median_cv')

    bp = ax.boxplot([pivot['Non-SnC'], pivot['SnC']],
                    positions=[0, 0.5], widths=0.3, showfliers=False, patch_artist=True,
                    boxprops=dict(linewidth=0.6),
                    medianprops=dict(color='k', linewidth=1.2),
                    whiskerprops=dict(color='#666', linewidth=0.5),
                    capprops=dict(color='#666', linewidth=0.5))

    bp['boxes'][0].set_facecolor(SNC_COLORS['Non-SnC']); bp['boxes'][0].set_alpha(0.7); bp['boxes'][0].set_edgecolor('#999')
    bp['boxes'][1].set_facecolor(SNC_COLORS['SnC']); bp['boxes'][1].set_alpha(0.7); bp['boxes'][1].set_edgecolor('#999')

    # Stats
    stat_row = [s for s in donor_cv_paired_stats if s['DEG_status'] == status]
    if stat_row:
        pval = stat_row[0]['wilcoxon_p']
        sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
        ax.text(0.03, 0.97, f'p={pval:.1e} {sig}',
                transform=ax.transAxes, fontsize=5.5, ha='left', va='top', color='#555')

    ax.set_xlim(-0.25, 0.75)
    ax.set_xticks([0, 0.5])
    ax.set_xticklabels(['Non-SnC', 'SnC'], fontsize=7)
    ax.set_title(f'{status} DEGs', fontsize=8, fontweight='bold',
                 color=DEG_COLORS.get(status, '#333'), pad=3)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.5); ax.spines['left'].set_linewidth(0.5)
    ax.tick_params(axis='y', labelsize=6)

    if ax == axes[0]:
        ax.set_ylabel('Median CV', fontsize=8)

plt.subplots_adjust(wspace=0.35)
plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_donor_deg_clean.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_donor_deg_clean.svg', dpi=300, bbox_inches='tight')
plt.show()
print("Saved clean donor-level CV figure")

---
## 10 · Gene-level CV, stratified by HVG status

**Why.** The second stratification, and a different kind of control. HVGs are
selected *because* they are variable, so their CV distribution is conditioned on
the outcome. Splitting by HVG status shows whether a CV difference holds among
genes that were not selected for variance.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# GENE-LEVEL CV: HVG-STRATIFIED
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("GENE-LEVEL CV (HVG-STRATIFIED)")
print("=" * 60)

snc_mask = adata_balanced.obs['is_senescent'].values
nonsnc_mask = ~snc_mask

X_snc = adata_balanced.X[snc_mask]
X_nonsnc = adata_balanced.X[nonsnc_mask]

if issparse(X_snc):
    X_snc = X_snc.toarray()
    X_nonsnc = X_nonsnc.toarray()

print(f"\n  SnC cells:     {X_snc.shape[0]:,}")
print(f"  Non-SnC cells: {X_nonsnc.shape[0]:,}")
print(f"  Genes:         {X_snc.shape[1]:,}")

# CV per gene
mean_snc = X_snc.mean(axis=0)
std_snc = X_snc.std(axis=0)
mean_nonsnc = X_nonsnc.mean(axis=0)
std_nonsnc = X_nonsnc.std(axis=0)

MIN_MEAN = 0.01
valid_mask = (mean_snc > MIN_MEAN) & (mean_nonsnc > MIN_MEAN)

cv_data = pd.DataFrame({
    'gene': adata_balanced.var_names,
    'mean_SnC': mean_snc,
    'mean_NonSnC': mean_nonsnc,
    'cv_SnC': np.where(valid_mask, std_snc / mean_snc, np.nan),
    'cv_NonSnC': np.where(valid_mask, std_nonsnc / mean_nonsnc, np.nan),
    'gene_status': adata_balanced.var['gene_status'].values,
}).dropna(subset=['cv_SnC', 'cv_NonSnC'])

cv_data['cv_diff'] = cv_data['cv_SnC'] - cv_data['cv_NonSnC']
cv_data['cv_ratio'] = cv_data['cv_SnC'] / cv_data['cv_NonSnC']

print(f"\n  Genes passing mean filter: {len(cv_data):,}")
print(f"    HVG:     {(cv_data['gene_status'] == 'HVG').sum():,}")
print(f"    Non-HVG: {(cv_data['gene_status'] == 'Non-HVG').sum():,}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CV STATISTICS BY HVG STATUS
# ════════════════════════════════════════════════════════════════════════════════

from scipy.stats import wilcoxon

print("=" * 60)
print("CV STATISTICS BY HVG STATUS")
print("=" * 60)

cv_stats = []

for status in ['HVG', 'Non-HVG']:
    subset = cv_data[cv_data['gene_status'] == status]
    if len(subset) < 10:
        print(f"  {status}: skipped (n={len(subset)})")
        continue

    stat, pval = wilcoxon(subset['cv_SnC'], subset['cv_NonSnC'])
    med_snc = subset['cv_SnC'].median()
    med_nonsnc = subset['cv_NonSnC'].median()
    med_diff = subset['cv_diff'].median()
    n_higher_snc = (subset['cv_diff'] > 0).sum()
    pct_higher = 100 * n_higher_snc / len(subset)
    direction = "SnC > Non-SnC" if med_diff > 0 else "Non-SnC > SnC"

    cv_stats.append({
        'gene_status': status, 'n_genes': len(subset),
        'median_cv_SnC': med_snc, 'median_cv_NonSnC': med_nonsnc,
        'median_cv_diff': med_diff, 'pct_higher_SnC': pct_higher,
        'wilcoxon_p': pval, 'direction': direction,
    })

    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
    print(f"\n  {status} (n={len(subset):,}):")
    print(f"    Median CV SnC:     {med_snc:.4f}")
    print(f"    Median CV Non-SnC: {med_nonsnc:.4f}")
    print(f"    ΔCV (SnC-NonSnC):  {med_diff:.4f}")
    print(f"    Higher in SnC:     {n_higher_snc:,} / {len(subset):,} ({pct_higher:.1f}%)")
    print(f"    Wilcoxon p:        {pval:.2e} {sig}")

# All genes
stat_all, pval_all = wilcoxon(cv_data['cv_SnC'], cv_data['cv_NonSnC'])
med_all_diff = cv_data['cv_diff'].median()
cv_stats.append({
    'gene_status': 'All', 'n_genes': len(cv_data),
    'median_cv_SnC': cv_data['cv_SnC'].median(),
    'median_cv_NonSnC': cv_data['cv_NonSnC'].median(),
    'median_cv_diff': med_all_diff,
    'pct_higher_SnC': 100 * (cv_data['cv_diff'] > 0).sum() / len(cv_data),
    'wilcoxon_p': pval_all,
    'direction': "SnC > Non-SnC" if med_all_diff > 0 else "Non-SnC > SnC",
})

print(f"\n  ALL genes (n={len(cv_data):,}):")
sig_all = '***' if pval_all < 0.001 else '**' if pval_all < 0.01 else '*' if pval_all < 0.05 else 'ns'
print(f"    ΔCV: {med_all_diff:.4f}, p={pval_all:.2e} {sig_all}")

df_cv_stats = pd.DataFrame(cv_stats)

---
## 11 · CV visualization, HVG-stratified and per study group

**Why.** The same distributions split by HVG status, then by study group — so a
difference driven by one group rather than being general becomes visible.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CV VISUALIZATION: HVG-STRATIFIED (PUBLICATION)
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("CV VISUALIZATION (HVG-STRATIFIED)")
print("=" * 60)

from scipy.stats import wilcoxon, mannwhitneyu
from matplotlib.patches import Patch

HVG_COLORS = {'HVG': '#E15759', 'Non-HVG': '#BDBDBD', 'All': '#808080'}

fig, axes = plt.subplots(2, 3, figsize=(13, 8))

# ── Panel A: Scatter ──
ax = axes[0, 0]
ax.grid(False)

for status, color in [('Non-HVG', '#BDBDBD'), ('HVG', '#E15759')]:
    sub = cv_data[cv_data['gene_status'] == status]
    alpha = 0.15 if status == 'Non-HVG' else 0.5
    size = 2 if status == 'Non-HVG' else 5
    ax.scatter(sub['cv_NonSnC'], sub['cv_SnC'], s=size, c=color, alpha=alpha,
               label=f'{status} ({len(sub):,})', rasterized=True)

lim_max = max(cv_data['cv_SnC'].quantile(0.99), cv_data['cv_NonSnC'].quantile(0.99))
ax.plot([0, lim_max], [0, lim_max], 'k--', lw=0.8, alpha=0.5)
ax.set_xlabel('CV (Non-SnC)', fontsize=10); ax.set_ylabel('CV (SnC)', fontsize=10)
ax.set_title('A. Gene-level CV', fontsize=11, fontweight='bold', loc='left')
ax.legend(fontsize=8, frameon=False, loc='upper left')
ax.set_xlim(0, lim_max); ax.set_ylim(0, lim_max)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=8)

# ── Panel B: ΔCV violin + stats ──
ax = axes[0, 1]
ax.grid(False)

vp_statuses = ['HVG', 'Non-HVG']
vp_data_b = [cv_data[cv_data['gene_status'] == s]['cv_diff'].values for s in vp_statuses]

parts_b = ax.violinplot(vp_data_b, positions=[0, 1], showextrema=False, showmedians=True)
for i, (body, status) in enumerate(zip(parts_b['bodies'], vp_statuses)):
    body.set_facecolor(HVG_COLORS[status]); body.set_alpha(0.7)
parts_b['cmedians'].set_color('black'); parts_b['cmedians'].set_linewidth(1.5)

ax.axhline(0, color='k', linestyle='--', lw=0.8, alpha=0.5)

for i, status in enumerate(vp_statuses):
    vals = cv_data[cv_data['gene_status'] == status]['cv_diff'].values
    if len(vals) > 10:
        _, p = wilcoxon(vals)
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        y_top = np.percentile(vals, 97)
        ax.text(i, y_top * 1.08, sig, ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks([0, 1]); ax.set_xticklabels(vp_statuses, fontsize=9)
ax.set_ylabel('ΔCV (SnC − Non-SnC)', fontsize=10)
ax.set_title('B. CV Difference', fontsize=11, fontweight='bold', loc='left')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=8)

# ── Panel C: Violin CV × senescence + stats ──
ax = axes[0, 2]
ax.grid(False)

positions_c = []
vp_data_c = []
vp_colors_c = []

for i, status in enumerate(['HVG', 'Non-HVG']):
    sub = cv_data[cv_data['gene_status'] == status]
    for j, (sen, col) in enumerate([('Non-SnC', 'cv_NonSnC'), ('SnC', 'cv_SnC')]):
        vals = sub[col].values
        if len(vals) > 0:
            positions_c.append(i * 2.5 + j * 0.9)
            vp_data_c.append(vals)
            vp_colors_c.append(SNC_COLORS[sen])

parts_c = ax.violinplot(vp_data_c, positions=positions_c, showextrema=False, showmedians=True, widths=0.7)
for body, color in zip(parts_c['bodies'], vp_colors_c):
    body.set_facecolor(color); body.set_alpha(0.7)
parts_c['cmedians'].set_color('black'); parts_c['cmedians'].set_linewidth(1.2)

for i, status in enumerate(['HVG', 'Non-HVG']):
    sub = cv_data[cv_data['gene_status'] == status]
    if len(sub) > 10:
        _, p = wilcoxon(sub['cv_SnC'], sub['cv_NonSnC'])
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        y_top = max(sub['cv_SnC'].quantile(0.97), sub['cv_NonSnC'].quantile(0.97))
        ax.text(i * 2.5 + 0.45, y_top * 1.08, sig, ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks([0.45, 2.95]); ax.set_xticklabels(['HVG', 'Non-HVG'], fontsize=9)
ax.set_ylabel('CV', fontsize=10)
ax.set_title('C. CV × Senescence', fontsize=11, fontweight='bold', loc='left')
ax.legend(handles=[Patch(facecolor=SNC_COLORS['Non-SnC'], label='Non-SnC'),
                   Patch(facecolor=SNC_COLORS['SnC'], label='SnC')],
          fontsize=8, frameon=False, loc='upper right')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=8)

# ── Panel D: % genes higher CV in SnC ──
ax = axes[1, 0]
ax.grid(False)

bar_data = df_cv_stats.copy()
bar_colors = [HVG_COLORS.get(s, '#808080') for s in bar_data['gene_status']]

bars = ax.bar(range(len(bar_data)), bar_data['pct_higher_SnC'], color=bar_colors, edgecolor='none', width=0.6)
ax.axhline(50, color='k', linestyle='--', lw=0.8, alpha=0.5)

for i, (_, row) in enumerate(bar_data.iterrows()):
    sig = '***' if row['wilcoxon_p'] < 0.001 else '**' if row['wilcoxon_p'] < 0.01 else '*' if row['wilcoxon_p'] < 0.05 else 'ns'
    ax.text(i, row['pct_higher_SnC'] + 1.5, sig, ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(range(len(bar_data)))
ax.set_xticklabels(bar_data['gene_status'], fontsize=9)
ax.set_ylabel('% Genes Higher CV in SnC', fontsize=10)
ax.set_ylim(0, 100)
ax.set_title('D. CV Direction', fontsize=11, fontweight='bold', loc='left')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=8)

# ═══════════════════════════════════════════════════════════════════════════════
# HELPER: Build study group violin with y-axis break
# ═══════════════════════════════════════════════════════════════════════════════

study_groups = [g for g in GROUP_ORDER if g in adata_balanced.obs[STUDY_GROUP_COL].values]

def compute_sg_cv(gene_idx):
    """Compute CV per study group × senescence for given gene indices."""
    results = []
    for sg in study_groups:
        sg_mask = adata_balanced.obs[STUDY_GROUP_COL] == sg
        for is_snc in [True, False]:
            cell_mask = sg_mask & (adata_balanced.obs['is_senescent'] == is_snc)
            if cell_mask.sum() < 20:
                continue
            X_sub = adata_balanced.X[cell_mask.values][:, gene_idx]
            if issparse(X_sub): X_sub = X_sub.toarray()
            means = X_sub.mean(axis=0)
            stds = X_sub.std(axis=0)
            valid = means > MIN_MEAN
            if valid.sum() < 5: continue
            cvs = stds[valid] / means[valid]
            for cv_val in cvs:
                results.append({'Study_Group': sg, 'Senescence': 'SnC' if is_snc else 'Non-SnC', 'CV': float(cv_val)})
    return pd.DataFrame(results)

def plot_sg_violin_broken(ax_top, ax_bot, sg_df, title, ylabel, break_low, break_high):
    """Plot violin with broken y-axis."""
    for a in [ax_top, ax_bot]:
        a.grid(False)
        a.spines['top'].set_visible(False)
        a.spines['right'].set_visible(False)

    positions_v = []
    vp_data_v = []
    vp_colors_v = []

    for i, sg in enumerate(study_groups):
        for j, sen in enumerate(['Non-SnC', 'SnC']):
            vals = sg_df[(sg_df['Study_Group'] == sg) & (sg_df['Senescence'] == sen)]['CV'].values
            if len(vals) > 5:
                positions_v.append(i * 2.2 + j * 0.8)
                vp_data_v.append(vals)
                vp_colors_v.append(SNC_COLORS[sen])

    if not vp_data_v:
        return

    for a in [ax_top, ax_bot]:
        parts = a.violinplot(vp_data_v, positions=positions_v, showextrema=False, showmedians=True, widths=0.6)
        for body, color in zip(parts['bodies'], vp_colors_v):
            body.set_facecolor(color); body.set_alpha(0.7)
        parts['cmedians'].set_color('black'); parts['cmedians'].set_linewidth(1.0)

    # Set y-limits for break
    ax_bot.set_ylim(0, break_low)
    ax_top.set_ylim(break_high, sg_df['CV'].quantile(0.99))

    # Break markers
    ax_bot.spines['top'].set_visible(False)
    ax_top.spines['bottom'].set_visible(False)
    ax_top.tick_params(bottom=False)

    d = 0.01
    kwargs = dict(transform=ax_top.transAxes, color='k', clip_on=False, linewidth=0.8)
    ax_top.plot((-d, +d), (-d, +d), **kwargs)
    ax_top.plot((1 - d, 1 + d), (-d, +d), **kwargs)
    kwargs.update(transform=ax_bot.transAxes)
    ax_bot.plot((-d, +d), (1 - d, 1 + d), **kwargs)
    ax_bot.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

    # Stats
    for i, sg in enumerate(study_groups):
        snc_v = sg_df[(sg_df['Study_Group'] == sg) & (sg_df['Senescence'] == 'SnC')]['CV'].values
        nonsnc_v = sg_df[(sg_df['Study_Group'] == sg) & (sg_df['Senescence'] == 'Non-SnC')]['CV'].values
        if len(snc_v) > 5 and len(nonsnc_v) > 5:
            _, p = mannwhitneyu(snc_v, nonsnc_v, alternative='two-sided')
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
            y_top = max(np.percentile(snc_v, 95), np.percentile(nonsnc_v, 95))
            target_ax = ax_top if y_top > break_high else ax_bot
            target_ax.text(i * 2.2 + 0.4, y_top * 1.03, sig, ha='center', va='bottom',
                          fontsize=7, fontweight='bold')

    # Labels on bottom axis only
    ax_bot.set_xticks([i * 2.2 + 0.4 for i in range(len(study_groups))])
    ax_bot.set_xticklabels(study_groups, fontsize=7, rotation=45, ha='right')
    ax_top.set_xticks([])

    ax_top.set_title(title, fontsize=11, fontweight='bold', loc='left')
    ax_bot.tick_params(labelsize=8)
    ax_top.tick_params(labelsize=8)

# ── Panel E: HVG CV by Study Group (with y-break) ──
# Since broken axis needs 2 subplots, we use the panel space creatively
# For simplicity, use single axis with clipped outliers instead

ax = axes[1, 1]
ax.grid(False)

hvg_genes_plot = cv_data[cv_data['gene_status'] == 'HVG']['gene'].values
hvg_idx_plot = [list(adata_balanced.var_names).index(g) for g in hvg_genes_plot if g in adata_balanced.var_names]
sg_cv_hvg_df = compute_sg_cv(hvg_idx_plot)

positions_e = []
vp_data_e = []
vp_colors_e = []

for i, sg in enumerate(study_groups):
    for j, sen in enumerate(['Non-SnC', 'SnC']):
        vals = sg_cv_hvg_df[(sg_cv_hvg_df['Study_Group'] == sg) & (sg_cv_hvg_df['Senescence'] == sen)]['CV'].values
        if len(vals) > 5:
            positions_e.append(i * 2.2 + j * 0.8)
            vp_data_e.append(vals)
            vp_colors_e.append(SNC_COLORS[sen])

if vp_data_e:
    parts_e = ax.violinplot(vp_data_e, positions=positions_e, showextrema=False, showmedians=True, widths=0.6)
    for body, color in zip(parts_e['bodies'], vp_colors_e):
        body.set_facecolor(color); body.set_alpha(0.7)
    parts_e['cmedians'].set_color('black'); parts_e['cmedians'].set_linewidth(1.0)

# Clip y to 99th percentile for readability
y_clip = sg_cv_hvg_df['CV'].quantile(0.99)
ax.set_ylim(0, y_clip * 1.1)

for i, sg in enumerate(study_groups):
    snc_v = sg_cv_hvg_df[(sg_cv_hvg_df['Study_Group'] == sg) & (sg_cv_hvg_df['Senescence'] == 'SnC')]['CV'].values
    nonsnc_v = sg_cv_hvg_df[(sg_cv_hvg_df['Study_Group'] == sg) & (sg_cv_hvg_df['Senescence'] == 'Non-SnC')]['CV'].values
    if len(snc_v) > 5 and len(nonsnc_v) > 5:
        _, p = mannwhitneyu(snc_v, nonsnc_v, alternative='two-sided')
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        y_top = min(max(np.percentile(snc_v, 95), np.percentile(nonsnc_v, 95)), y_clip)
        ax.text(i * 2.2 + 0.4, y_top * 1.03, sig, ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xticks([i * 2.2 + 0.4 for i in range(len(study_groups))])
ax.set_xticklabels(study_groups, fontsize=7, rotation=45, ha='right')
ax.set_ylabel('CV (HVGs)', fontsize=10)
ax.set_title('E. HVG CV by Study Group', fontsize=11, fontweight='bold', loc='left')
ax.legend(handles=[Patch(facecolor=SNC_COLORS['Non-SnC'], label='Non-SnC'),
                   Patch(facecolor=SNC_COLORS['SnC'], label='SnC')],
          fontsize=8, frameon=False, loc='upper right')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=8)

# ── Panel F: Non-HVG CV by Study Group ──
ax = axes[1, 2]
ax.grid(False)

nonhvg_genes_plot = cv_data[cv_data['gene_status'] == 'Non-HVG']['gene'].values
nonhvg_idx_plot = [list(adata_balanced.var_names).index(g) for g in nonhvg_genes_plot if g in adata_balanced.var_names]
sg_cv_nonhvg_df = compute_sg_cv(nonhvg_idx_plot)

positions_f = []
vp_data_f = []
vp_colors_f = []

for i, sg in enumerate(study_groups):
    for j, sen in enumerate(['Non-SnC', 'SnC']):
        vals = sg_cv_nonhvg_df[(sg_cv_nonhvg_df['Study_Group'] == sg) & (sg_cv_nonhvg_df['Senescence'] == sen)]['CV'].values
        if len(vals) > 5:
            positions_f.append(i * 2.2 + j * 0.8)
            vp_data_f.append(vals)
            vp_colors_f.append(SNC_COLORS[sen])

if vp_data_f:
    parts_f = ax.violinplot(vp_data_f, positions=positions_f, showextrema=False, showmedians=True, widths=0.6)
    for body, color in zip(parts_f['bodies'], vp_colors_f):
        body.set_facecolor(color); body.set_alpha(0.7)
    parts_f['cmedians'].set_color('black'); parts_f['cmedians'].set_linewidth(1.0)

y_clip_f = sg_cv_nonhvg_df['CV'].quantile(0.99)
ax.set_ylim(0, y_clip_f * 1.1)

for i, sg in enumerate(study_groups):
    snc_v = sg_cv_nonhvg_df[(sg_cv_nonhvg_df['Study_Group'] == sg) & (sg_cv_nonhvg_df['Senescence'] == 'SnC')]['CV'].values
    nonsnc_v = sg_cv_nonhvg_df[(sg_cv_nonhvg_df['Study_Group'] == sg) & (sg_cv_nonhvg_df['Senescence'] == 'Non-SnC')]['CV'].values
    if len(snc_v) > 5 and len(nonsnc_v) > 5:
        _, p = mannwhitneyu(snc_v, nonsnc_v, alternative='two-sided')
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        y_top = min(max(np.percentile(snc_v, 95), np.percentile(nonsnc_v, 95)), y_clip_f)
        ax.text(i * 2.2 + 0.4, y_top * 1.03, sig, ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xticks([i * 2.2 + 0.4 for i in range(len(study_groups))])
ax.set_xticklabels(study_groups, fontsize=7, rotation=45, ha='right')
ax.set_ylabel('CV (Non-HVGs)', fontsize=10)
ax.set_title('F. Non-HVG CV by Study Group', fontsize=11, fontweight='bold', loc='left')
ax.legend(handles=[Patch(facecolor=SNC_COLORS['Non-SnC'], label='Non-SnC'),
                   Patch(facecolor=SNC_COLORS['SnC'], label='SnC')],
          fontsize=8, frameon=False, loc='upper right')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_hvg_stratified.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_hvg_stratified.svg', dpi=300, bbox_inches='tight')
plt.show()
print("Saved HVG-stratified CV figure (6 panels)")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CV SUMMARY TABLES
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("CV SUMMARY TABLES")
print("=" * 60)

# ── Table 1: Overall CV stats by HVG status ──
print(f"\n{'─'*80}")
print("TABLE 1: CV BY HVG STATUS")
print(f"{'─'*80}")
print(f"\n  {'Status':<10} {'N genes':>8} {'CV SnC':>10} {'CV NonSnC':>10} {'ΔCV':>8} {'%Higher':>8} {'p':>12} {'Sig':<4}")
print(f"  {'─'*72}")

for _, row in df_cv_stats.iterrows():
    sig = '***' if row['wilcoxon_p'] < 0.001 else '**' if row['wilcoxon_p'] < 0.01 else '*' if row['wilcoxon_p'] < 0.05 else 'ns'
    print(f"  {row['gene_status']:<10} {row['n_genes']:>8,} {row['median_cv_SnC']:>10.4f} "
          f"{row['median_cv_NonSnC']:>10.4f} {row['median_cv_diff']:>8.4f} "
          f"{row['pct_higher_SnC']:>7.1f}% {row['wilcoxon_p']:>12.2e} {sig:<4}")

# ── Table 2: CV by Study Group (HVGs) ──
print(f"\n{'─'*80}")
print("TABLE 2: HVG CV BY STUDY GROUP")
print(f"{'─'*80}")
print(f"\n  {'Study Group':<15} {'N SnC':>8} {'N NonSnC':>8} {'CV SnC':>10} {'CV NonSnC':>10} {'ΔCV':>8} {'p':>12} {'Sig':<4}")
print(f"  {'─'*78}")

sg_hvg_table = []
for sg in study_groups:
    snc_v = sg_cv_hvg_df[(sg_cv_hvg_df['Study_Group'] == sg) & (sg_cv_hvg_df['Senescence'] == 'SnC')]['CV'].values
    nonsnc_v = sg_cv_hvg_df[(sg_cv_hvg_df['Study_Group'] == sg) & (sg_cv_hvg_df['Senescence'] == 'Non-SnC')]['CV'].values

    if len(snc_v) > 5 and len(nonsnc_v) > 5:
        _, p = mannwhitneyu(snc_v, nonsnc_v, alternative='two-sided')
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        med_s = np.median(snc_v)
        med_n = np.median(nonsnc_v)
        delta = med_s - med_n

        sg_hvg_table.append({
            'Study_Group': sg, 'n_SnC': len(snc_v), 'n_NonSnC': len(nonsnc_v),
            'median_cv_SnC': med_s, 'median_cv_NonSnC': med_n,
            'delta_cv': delta, 'p_value': p,
        })

        print(f"  {sg:<15} {len(snc_v):>8,} {len(nonsnc_v):>8,} {med_s:>10.4f} "
              f"{med_n:>10.4f} {delta:>8.4f} {p:>12.2e} {sig:<4}")
    else:
        print(f"  {sg:<15} {'insufficient data':>50}")

# ── Table 3: CV by Study Group (Non-HVGs) ──
print(f"\n{'─'*80}")
print("TABLE 3: NON-HVG CV BY STUDY GROUP")
print(f"{'─'*80}")
print(f"\n  {'Study Group':<15} {'N SnC':>8} {'N NonSnC':>8} {'CV SnC':>10} {'CV NonSnC':>10} {'ΔCV':>8} {'p':>12} {'Sig':<4}")
print(f"  {'─'*78}")

sg_nonhvg_table = []
for sg in study_groups:
    snc_v = sg_cv_nonhvg_df[(sg_cv_nonhvg_df['Study_Group'] == sg) & (sg_cv_nonhvg_df['Senescence'] == 'SnC')]['CV'].values
    nonsnc_v = sg_cv_nonhvg_df[(sg_cv_nonhvg_df['Study_Group'] == sg) & (sg_cv_nonhvg_df['Senescence'] == 'Non-SnC')]['CV'].values

    if len(snc_v) > 5 and len(nonsnc_v) > 5:
        _, p = mannwhitneyu(snc_v, nonsnc_v, alternative='two-sided')
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        med_s = np.median(snc_v)
        med_n = np.median(nonsnc_v)
        delta = med_s - med_n

        sg_nonhvg_table.append({
            'Study_Group': sg, 'n_SnC': len(snc_v), 'n_NonSnC': len(nonsnc_v),
            'median_cv_SnC': med_s, 'median_cv_NonSnC': med_n,
            'delta_cv': delta, 'p_value': p,
        })

        print(f"  {sg:<15} {len(snc_v):>8,} {len(nonsnc_v):>8,} {med_s:>10.4f} "
              f"{med_n:>10.4f} {delta:>8.4f} {p:>12.2e} {sig:<4}")
    else:
        print(f"  {sg:<15} {'insufficient data':>50}")

# ── Save tables ──
df_cv_stats.to_csv(RESULTS_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_overall_stats.csv', index=False)
pd.DataFrame(sg_hvg_table).to_csv(RESULTS_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_hvg_by_studygroup.csv', index=False)
pd.DataFrame(sg_nonhvg_table).to_csv(RESULTS_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_nonhvg_by_studygroup.csv', index=False)

print(f"\n✓ Saved tables to {RESULTS_DIR}")

---
## 12 · Donor-level CV, HVG-stratified

**Why.** The donor-level version of section 10, for the same reason as
section 09.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# DONOR-LEVEL CV: HVG-STRATIFIED
# ════════════════════════════════════════════════════════════════════════════════

from scipy.stats import wilcoxon

print("=" * 60)
print("DONOR-LEVEL CV (HVG-STRATIFIED)")
print("=" * 60)

# Get gene lists from cv_data
hvg_genes = cv_data[cv_data['gene_status'] == 'HVG']['gene'].values
nonhvg_genes = cv_data[cv_data['gene_status'] == 'Non-HVG']['gene'].values
print(f"  HVG genes: {len(hvg_genes)}")
print(f"  Non-HVG genes: {len(nonhvg_genes)}")

hvg_gene_idx = [list(adata_balanced.var_names).index(g) for g in hvg_genes if g in adata_balanced.var_names]
nonhvg_gene_idx = [list(adata_balanced.var_names).index(g) for g in nonhvg_genes if g in adata_balanced.var_names]

gene_sets = {'HVG': hvg_gene_idx, 'Non-HVG': nonhvg_gene_idx}

donor_cv_hvg = []

for donor in adata_balanced.obs[SAMPLE_COL].unique():
    donor_mask = adata_balanced.obs[SAMPLE_COL] == donor
    study_group = adata_balanced.obs.loc[donor_mask, STUDY_GROUP_COL].iloc[0]

    for is_snc in [True, False]:
        cell_mask = donor_mask & (adata_balanced.obs['is_senescent'] == is_snc)
        n_cells = cell_mask.sum()
        if n_cells < MIN_CELLS_PER_GROUP:
            continue

        X_sub = adata_balanced.X[cell_mask.values]
        if issparse(X_sub): X_sub = X_sub.toarray()

        for status, idx in gene_sets.items():
            if len(idx) < 10: continue
            X_genes = X_sub[:, idx]
            means = X_genes.mean(axis=0)
            stds = X_genes.std(axis=0)
            valid = means > MIN_MEAN
            if valid.sum() < 5: continue
            cvs = stds[valid] / means[valid]

            donor_cv_hvg.append({
                'donor': donor, 'Study_Group': study_group,
                'Senescence': 'SnC' if is_snc else 'Non-SnC',
                'gene_status': status, 'n_cells': n_cells,
                'median_cv': float(np.median(cvs)), 'mean_cv': float(cvs.mean()),
                'n_genes': int(valid.sum()),
            })

donor_cv_hvg = pd.DataFrame(donor_cv_hvg)
print(f"\n  Total observations: {len(donor_cv_hvg)}")

# Paired tests
print(f"\n  {'Status':<10} {'n_paired':>8} {'SnC CV':>10} {'NonSnC CV':>10} {'p':>10} {'Dir':<15}")
print(f"  {'─'*60}")

donor_cv_paired_stats = []

for status in ['HVG', 'Non-HVG']:
    sub = donor_cv_hvg[donor_cv_hvg['gene_status'] == status]
    donors_s = set(sub[sub['Senescence'] == 'SnC']['donor'])
    donors_n = set(sub[sub['Senescence'] == 'Non-SnC']['donor'])
    paired = sorted(donors_s & donors_n)

    if len(paired) < 5:
        print(f"  {status:<10} {'<5':>8}")
        continue

    pivot = sub[sub['donor'].isin(paired)].pivot(index='donor', columns='Senescence', values='median_cv')
    stat, pval = wilcoxon(pivot['SnC'], pivot['Non-SnC'])
    med_s = pivot['SnC'].median()
    med_n = pivot['Non-SnC'].median()
    direction = "SnC > Non-SnC" if med_s > med_n else "Non-SnC > SnC"

    donor_cv_paired_stats.append({
        'gene_status': status, 'n_paired': len(paired),
        'median_cv_SnC': med_s, 'median_cv_NonSnC': med_n,
        'wilcoxon_p': pval, 'direction': direction,
    })

    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
    print(f"  {status:<10} {len(paired):>8} {med_s:>10.4f} {med_n:>10.4f} {pval:>10.2e} {direction:<15} {sig}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# DONOR-LEVEL CV FIGURE (HVG-STRATIFIED) — PAIRED
# ════════════════════════════════════════════════════════════════════════════════

import matplotlib.cm as cm

print("=" * 60)
print("DONOR-LEVEL CV FIGURE")
print("=" * 60)

statuses_plot = ['HVG', 'Non-HVG']

fig, axes = plt.subplots(1, 2, figsize=(5.5, 2.8))

for ax, status in zip(axes, statuses_plot):
    ax.grid(False)
    sub = donor_cv_hvg[donor_cv_hvg['gene_status'] == status]

    donors_s = set(sub[sub['Senescence'] == 'SnC']['donor'])
    donors_n = set(sub[sub['Senescence'] == 'Non-SnC']['donor'])
    paired = sorted(donors_s & donors_n)

    if len(paired) < 5:
        ax.text(0.5, 0.5, f'{status}\nn < 5', ha='center', va='center',
                transform=ax.transAxes, fontsize=8, color='gray')
        continue

    pivot = sub[sub['donor'].isin(paired)].pivot(index='donor', columns='Senescence', values='median_cv')

    donor_cmap = cm.get_cmap('tab20', len(paired))
    donor_colors = {d: donor_cmap(i) for i, d in enumerate(paired)}

    bp = ax.boxplot([pivot['Non-SnC'], pivot['SnC']],
                    positions=[0, 0.6], widths=0.2, showfliers=False, patch_artist=True,
                    boxprops=dict(facecolor='white', edgecolor='#999', linewidth=0.6),
                    medianprops=dict(color='k', linewidth=1.2),
                    whiskerprops=dict(color='#999', linewidth=0.5),
                    capprops=dict(color='#999', linewidth=0.5))

    for donor in pivot.index:
        c = donor_colors[donor]
        jx = np.random.uniform(-0.04, 0.04)
        ax.plot([0 + jx, 0.6 + jx],
                [pivot.loc[donor, 'Non-SnC'], pivot.loc[donor, 'SnC']],
                c=c, alpha=0.4, lw=0.6, zorder=1)
        ax.scatter(0 + jx, pivot.loc[donor, 'Non-SnC'],
                   c=[c], s=10, edgecolor='white', linewidth=0.2, zorder=3)
        ax.scatter(0.6 + jx, pivot.loc[donor, 'SnC'],
                   c=[c], s=10, edgecolor='white', linewidth=0.2, zorder=3)

    stat_row = [s for s in donor_cv_paired_stats if s['gene_status'] == status]
    if stat_row:
        pval = stat_row[0]['wilcoxon_p']
        sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
        ax.text(0.03, 0.97, f'p={pval:.1e} {sig}\nn={len(paired)}',
                transform=ax.transAxes, fontsize=5.5, ha='left', va='top', color='#555')

    ax.set_xlim(-0.25, 0.85)
    ax.set_xticks([0, 0.6])
    ax.set_xticklabels(['Non-SnC', 'SnC'], fontsize=7)
    ax.set_title(f'{status}', fontsize=8, fontweight='bold',
                 color=HVG_COLORS.get(status, '#333'), pad=3)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.5); ax.spines['left'].set_linewidth(0.5)
    ax.tick_params(axis='y', labelsize=6)

    if ax == axes[0]:
        ax.set_ylabel('Median CV', fontsize=8)

plt.subplots_adjust(wspace=0.35)
plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_donor_hvg.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_donor_hvg.svg', dpi=300, bbox_inches='tight')
plt.show()
print("Saved donor-level HVG-stratified CV figure")

---
## 13 · Save CV results

**Why.** Writes the CV tables so the figures below and any downstream reuse read
from a saved artifact rather than re-deriving.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SAVE CV RESULTS
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("SAVING CV RESULTS")
print("=" * 60)

cv_data.to_csv(RESULTS_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_genelevel.csv', index=False)
donor_cv_hvg.to_csv(RESULTS_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_donorlevel_hvg.csv', index=False)
df_cv_stats.to_csv(RESULTS_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_stats.csv', index=False)

print(f"\n  Saved to: {RESULTS_DIR}")
for f in sorted(RESULTS_DIR.glob('*cv*')):
    print(f"    {f.name}")

print("\n✓ CV analysis complete")

In [ ]:
print("=" * 60)
print("UMI CHECK: SnC vs Non-SnC")
print("=" * 60)

from scipy.stats import mannwhitneyu

for label in ['SnC', 'Non-SnC']:
    mask = adata_balanced.obs['is_senescent'] == (label == 'SnC')
    med = adata_balanced.obs.loc[mask, 'total_counts'].median()
    mean = adata_balanced.obs.loc[mask, 'total_counts'].mean()
    print(f"\n  {label}:")
    print(f"    Median UMI: {med:,.0f}")
    print(f"    Mean UMI:   {mean:,.0f}")
    print(f"    N cells:    {mask.sum():,}")

snc_umi = adata_balanced.obs[adata_balanced.obs['is_senescent']]['total_counts'].values
nonsnc_umi = adata_balanced.obs[~adata_balanced.obs['is_senescent']]['total_counts'].values

stat, pval = mannwhitneyu(snc_umi, nonsnc_umi, alternative='two-sided')
ratio = np.median(snc_umi) / np.median(nonsnc_umi)

print(f"\n  Ratio (SnC/Non-SnC): {ratio:.2f}x")
print(f"  Mann-Whitney p: {pval:.2e}")

if ratio > 1.2:
    print(f"\n  ⚠ SnC has {ratio:.1f}x higher UMI — CV difference may be partly technical")
elif ratio < 0.8:
    print(f"\n  SnC has lower UMI — CV difference is UNDERSTATED (real effect even stronger)")
else:
    print(f"\n  ✓ Similar library sizes — CV difference is biological")

---
## 14 · Depth-normalized CV

**Why.** The direct answer to the depth confound measured in section 05.
Recomputes CV after normalizing for sequencing depth, so the comparison is not
carried by senescent cells simply having more reads.

**Read this against section 07.** If the raw CV difference survives depth
normalization, depth was not the explanation. If it collapses, it was.

**Display.** Depth-normalized CV per study group, then violin plus swarm so
individual genes stay visible rather than being summarized away.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# DEPTH-NORMALIZED CV + PER STUDY GROUP
# ════════════════════════════════════════════════════════════════════════════════

from sklearn.linear_model import LinearRegression
from scipy.stats import wilcoxon, mannwhitneyu
from matplotlib.patches import Patch

print("=" * 60)
print("DEPTH-NORMALIZED CV")
print("=" * 60)

# Set expression matrix
if 'lognorm' in adata_balanced.layers:
    adata_balanced.X = adata_balanced.layers['lognorm'].copy()
    print(f"  Set X = lognorm layer")
elif 'log_normalized' in adata_balanced.layers:
    adata_balanced.X = adata_balanced.layers['log_normalized'].copy()
    print(f"  Set X = log_normalized layer")
else:
    x_max = adata_balanced.X.max()
    if 4 < x_max < 15:
        print(f"  X already log-normalized (range 0–{x_max:.2f}), using as-is")
    else:
        print(f"  ⚠ X range 0–{x_max:.2f} — may need normalization")

# Compute total_counts if missing
if 'total_counts' not in adata_balanced.obs.columns:
    if 'counts' in adata_balanced.layers:
        counts = adata_balanced.layers['counts']
        if issparse(counts):
            adata_balanced.obs['total_counts'] = np.array(counts.sum(axis=1)).flatten()
        else:
            adata_balanced.obs['total_counts'] = counts.sum(axis=1)
        print(f"  Computed total_counts from counts layer")
    else:
        print(f"  ⚠ No total_counts and no counts layer")

depth_cv = []

for donor in adata_balanced.obs[SAMPLE_COL].unique():
    donor_mask = adata_balanced.obs[SAMPLE_COL] == donor
    study_group = adata_balanced.obs.loc[donor_mask, STUDY_GROUP_COL].iloc[0]

    for is_snc in [True, False]:
        cell_mask = donor_mask & (adata_balanced.obs['is_senescent'] == is_snc)
        n_cells = cell_mask.sum()
        if n_cells < MIN_CELLS_PER_GROUP:
            continue

        X_sub = adata_balanced.X[cell_mask.values]
        if issparse(X_sub): X_sub = X_sub.toarray()

        means = X_sub.mean(axis=0)
        stds = X_sub.std(axis=0)
        valid = means > MIN_MEAN
        if valid.sum() < 5: continue
        cvs = stds[valid] / means[valid]

        median_umi = adata_balanced.obs.loc[cell_mask, 'total_counts'].median()

        depth_cv.append({
            'donor': donor, 'Study_Group': study_group,
            'Senescence': 'SnC' if is_snc else 'Non-SnC',
            'median_cv': float(np.median(cvs)),
            'log10_median_umi': float(np.log10(median_umi + 1)),
            'n_cells': n_cells,
        })

depth_cv_df = pd.DataFrame(depth_cv)

if len(depth_cv_df) < 4:
    print(f"\n  ⚠ Only {len(depth_cv_df)} observations — insufficient for analysis")
    print(f"  Consider lowering MIN_CELLS_PER_GROUP (currently {MIN_CELLS_PER_GROUP})")
else:
    # Fit: CV ~ log10(UMI)
    reg = LinearRegression()
    reg.fit(depth_cv_df[['log10_median_umi']].values, depth_cv_df['median_cv'].values)
    depth_cv_df['cv_predicted'] = reg.predict(depth_cv_df[['log10_median_umi']].values)
    depth_cv_df['cv_residual'] = depth_cv_df['median_cv'] - depth_cv_df['cv_predicted']

    r2 = reg.score(depth_cv_df[['log10_median_umi']].values, depth_cv_df['median_cv'].values)
    print(f"  CV ~ log10(UMI): R² = {r2:.3f}")
    print(f"  Slope: {reg.coef_[0]:.4f}, Intercept: {reg.intercept_:.4f}")

    # Paired tests
    donors_both = set(depth_cv_df[depth_cv_df['Senescence'] == 'SnC']['donor']) & \
                  set(depth_cv_df[depth_cv_df['Senescence'] == 'Non-SnC']['donor'])
    paired_df = depth_cv_df[depth_cv_df['donor'].isin(donors_both)]

    if len(donors_both) < 5:
        print(f"\n  ⚠ Only {len(donors_both)} paired donors — insufficient for Wilcoxon")
    else:
        pivot_raw = paired_df.pivot(index='donor', columns='Senescence', values='median_cv')
        pivot_resid = paired_df.pivot(index='donor', columns='Senescence', values='cv_residual')

        stat_raw, p_raw = wilcoxon(pivot_raw['SnC'], pivot_raw['Non-SnC'])
        stat_resid, p_resid = wilcoxon(pivot_resid['SnC'], pivot_resid['Non-SnC'])

        print(f"\n  OVERALL (n={len(donors_both)} paired donors)")
        print(f"  {'Metric':<25} {'SnC':>10} {'Non-SnC':>10} {'p':>12} {'Sig':<4}")
        print(f"  {'─'*62}")
        for label, pivot, p in [('Raw CV', pivot_raw, p_raw), ('Depth-normalized CV', pivot_resid, p_resid)]:
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
            print(f"  {label:<25} {pivot['SnC'].median():>10.4f} {pivot['Non-SnC'].median():>10.4f} {p:>12.2e} {sig:<4}")

        if p_resid < 0.05:
            direction = "SnC > Non-SnC" if pivot_resid['SnC'].median() > pivot_resid['Non-SnC'].median() else "Non-SnC > SnC"
            print(f"\n  ✓ CV difference PERSISTS after depth correction ({direction})")
        else:
            print(f"\n  ⚠ CV difference DISAPPEARS after depth correction")

        # ── Per Study Group ──
        study_groups = [g for g in GROUP_ORDER if g in depth_cv_df['Study_Group'].values]

        print(f"\n  {'─'*98}")
        print(f"  PER STUDY GROUP (Depth-normalized CV)")
        print(f"  {'─'*98}")
        print(f"\n  {'Study Group':<25} {'n':>4} {'Raw SnC':>10} {'Raw NonSnC':>10} {'p_raw':>10} {'Resid SnC':>10} {'Resid NonSnC':>12} {'p_resid':>10} {'Sig':<4}")
        print(f"  {'─'*98}")

        sg_depth_results = []

        for sg in study_groups:
            sg_paired = paired_df[paired_df['Study_Group'] == sg]
            sg_donors = set(sg_paired[sg_paired['Senescence'] == 'SnC']['donor']) & \
                        set(sg_paired[sg_paired['Senescence'] == 'Non-SnC']['donor'])

            if len(sg_donors) < 5:
                print(f"  {sg:<25} {'<5 paired':>4}")
                continue

            sg_data = sg_paired[sg_paired['donor'].isin(sg_donors)]
            piv_raw = sg_data.pivot(index='donor', columns='Senescence', values='median_cv')
            piv_res = sg_data.pivot(index='donor', columns='Senescence', values='cv_residual')

            _, p_r = wilcoxon(piv_raw['SnC'], piv_raw['Non-SnC'])
            _, p_d = wilcoxon(piv_res['SnC'], piv_res['Non-SnC'])

            sig = '***' if p_d < 0.001 else '**' if p_d < 0.01 else '*' if p_d < 0.05 else 'ns'

            sg_depth_results.append({
                'Study_Group': sg, 'n_paired': len(sg_donors),
                'raw_cv_SnC': piv_raw['SnC'].median(), 'raw_cv_NonSnC': piv_raw['Non-SnC'].median(),
                'p_raw': p_r,
                'resid_cv_SnC': piv_res['SnC'].median(), 'resid_cv_NonSnC': piv_res['Non-SnC'].median(),
                'p_resid': p_d,
            })

            print(f"  {sg:<25} {len(sg_donors):>4} {piv_raw['SnC'].median():>10.4f} {piv_raw['Non-SnC'].median():>10.4f} "
                  f"{p_r:>10.2e} {piv_res['SnC'].median():>10.4f} {piv_res['Non-SnC'].median():>12.4f} {p_d:>10.2e} {sig:<4}")

        pd.DataFrame(sg_depth_results).to_csv(RESULTS_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_depth_by_studygroup.csv', index=False)

        # ── Filter study groups with data for plotting ──
        study_groups_plot = [sg for sg in study_groups
                            if len(depth_cv_df[depth_cv_df['Study_Group'] == sg]) >= 6]

        # ── Figure: 2×2 ──
        fig, axes = plt.subplots(2, 2, figsize=(11, 9))

        def style_ax(ax):
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_linewidth(0.8)
                spine.set_color('#333')
            ax.tick_params(labelsize=9, width=0.8, length=4)

        # Panel A: CV vs UMI
        ax = axes[0, 0]
        ax.grid(False)
        for sen, color in [('Non-SnC', SNC_COLORS['Non-SnC']), ('SnC', SNC_COLORS['SnC'])]:
            sub = depth_cv_df[depth_cv_df['Senescence'] == sen]
            ax.scatter(sub['log10_median_umi'], sub['median_cv'], c=color, s=20, alpha=0.6, label=sen)

        x_line = np.linspace(depth_cv_df['log10_median_umi'].min(), depth_cv_df['log10_median_umi'].max(), 100)
        ax.plot(x_line, reg.predict(x_line.reshape(-1, 1)), 'k--', lw=1, alpha=0.5)

        ax.set_xlabel('log₁₀(Median UMI)', fontsize=10)
        ax.set_ylabel('Median CV', fontsize=10)
        ax.set_title('A. CV vs Library Depth', fontsize=11, fontweight='bold', loc='left')
        ax.text(0.03, 0.97, f'R²={r2:.3f}', transform=ax.transAxes, fontsize=9, ha='left', va='top', color='#333')
        ax.legend(fontsize=9, frameon=False)
        style_ax(ax)

        # Panel B: Raw vs Depth-normalized
        ax = axes[0, 1]
        ax.grid(False)

        bp_positions = [0, 0.6, 1.8, 2.4]
        bp_data = [pivot_raw['Non-SnC'], pivot_raw['SnC'], pivot_resid['Non-SnC'], pivot_resid['SnC']]
        bp_colors = [SNC_COLORS['Non-SnC'], SNC_COLORS['SnC'], SNC_COLORS['Non-SnC'], SNC_COLORS['SnC']]

        bp = ax.boxplot(bp_data, positions=bp_positions, widths=0.4, showfliers=False, patch_artist=True,
                        boxprops=dict(linewidth=0.6),
                        medianprops=dict(color='k', linewidth=1.2),
                        whiskerprops=dict(color='#666', linewidth=0.5),
                        capprops=dict(color='#666', linewidth=0.5))

        for box, color in zip(bp['boxes'], bp_colors):
            box.set_facecolor(color); box.set_alpha(0.7); box.set_edgecolor('#999')

        sig_r = '***' if p_raw < 0.001 else '**' if p_raw < 0.01 else '*' if p_raw < 0.05 else 'ns'
        sig_d = '***' if p_resid < 0.001 else '**' if p_resid < 0.01 else '*' if p_resid < 0.05 else 'ns'
        ax.text(0.3, 0.97, sig_r, ha='center', va='top', fontsize=9, fontweight='bold', transform=ax.transAxes)
        ax.text(0.75, 0.97, sig_d, ha='center', va='top', fontsize=9, fontweight='bold', transform=ax.transAxes)

        ax.set_xticks([0.3, 2.1])
        ax.set_xticklabels(['Raw CV', 'Depth-normalized'], fontsize=9)
        ax.set_ylabel('CV / Residual CV', fontsize=10)
        ax.set_title('B. Raw vs Corrected', fontsize=11, fontweight='bold', loc='left')
        ax.legend(handles=[Patch(facecolor=SNC_COLORS['Non-SnC'], label='Non-SnC'),
                           Patch(facecolor=SNC_COLORS['SnC'], label='SnC')],
                  fontsize=8, frameon=False, loc='upper right')
        style_ax(ax)

        # Panel C: Depth-normalized CV by Study Group (violin)
        ax = axes[1, 0]
        ax.grid(False)

        if study_groups_plot:
            positions_c = []
            vp_data_c = []
            vp_colors_c = []

            for i, sg in enumerate(study_groups_plot):
                sg_data = depth_cv_df[depth_cv_df['Study_Group'] == sg]
                for j, sen in enumerate(['Non-SnC', 'SnC']):
                    vals = sg_data[sg_data['Senescence'] == sen]['cv_residual'].values
                    if len(vals) > 3:
                        positions_c.append(i * 2.2 + j * 0.8)
                        vp_data_c.append(vals)
                        vp_colors_c.append(SNC_COLORS[sen])

            if vp_data_c:
                parts = ax.violinplot(vp_data_c, positions=positions_c, showextrema=False, showmedians=True, widths=0.6)
                for body, color in zip(parts['bodies'], vp_colors_c):
                    body.set_facecolor(color); body.set_alpha(0.7)
                parts['cmedians'].set_color('black'); parts['cmedians'].set_linewidth(1.0)

            ax.axhline(0, color='k', linestyle='--', lw=0.8, alpha=0.3)

            for i, sg in enumerate(study_groups_plot):
                sg_res = [r for r in sg_depth_results if r['Study_Group'] == sg]
                if sg_res:
                    p = sg_res[0]['p_resid']
                    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
                    ax.text(i * 2.2 + 0.4, ax.get_ylim()[1] * 0.95, sig, ha='center', va='top',
                            fontsize=7, fontweight='bold')

            ax.set_xticks([i * 2.2 + 0.4 for i in range(len(study_groups_plot))])
            ax.set_xticklabels(study_groups_plot, fontsize=7, rotation=45, ha='right')
        else:
            ax.text(0.5, 0.5, 'Insufficient data\nper study group', ha='center', va='center',
                    transform=ax.transAxes, fontsize=10, color='gray')

        ax.set_ylabel('Depth-normalized CV (residual)', fontsize=10)
        ax.set_title('C. Corrected CV by Study Group', fontsize=11, fontweight='bold', loc='left')
        ax.legend(handles=[Patch(facecolor=SNC_COLORS['Non-SnC'], label='Non-SnC'),
                           Patch(facecolor=SNC_COLORS['SnC'], label='SnC')],
                  fontsize=7, frameon=False, loc='upper right')
        style_ax(ax)

        # Panel D: UMI by Study Group
        ax = axes[1, 1]
        ax.grid(False)

        if study_groups_plot:
            positions_d = []
            vp_data_d = []
            vp_colors_d = []

            for i, sg in enumerate(study_groups_plot):
                sg_data = depth_cv_df[depth_cv_df['Study_Group'] == sg]
                for j, sen in enumerate(['Non-SnC', 'SnC']):
                    vals = sg_data[sg_data['Senescence'] == sen]['log10_median_umi'].values
                    if len(vals) > 3:
                        positions_d.append(i * 2.2 + j * 0.8)
                        vp_data_d.append(vals)
                        vp_colors_d.append(SNC_COLORS[sen])

            if vp_data_d:
                parts = ax.violinplot(vp_data_d, positions=positions_d, showextrema=False, showmedians=True, widths=0.6)
                for body, color in zip(parts['bodies'], vp_colors_d):
                    body.set_facecolor(color); body.set_alpha(0.7)
                parts['cmedians'].set_color('black'); parts['cmedians'].set_linewidth(1.0)

            for i, sg in enumerate(study_groups_plot):
                sg_data = depth_cv_df[depth_cv_df['Study_Group'] == sg]
                snc_u = sg_data[sg_data['Senescence'] == 'SnC']['log10_median_umi'].values
                nonsnc_u = sg_data[sg_data['Senescence'] == 'Non-SnC']['log10_median_umi'].values
                if len(snc_u) > 3 and len(nonsnc_u) > 3:
                    _, p = mannwhitneyu(snc_u, nonsnc_u, alternative='two-sided')
                    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
                    ax.text(i * 2.2 + 0.4, ax.get_ylim()[1] * 0.95, sig, ha='center', va='top',
                            fontsize=7, fontweight='bold')

            ax.set_xticks([i * 2.2 + 0.4 for i in range(len(study_groups_plot))])
            ax.set_xticklabels(study_groups_plot, fontsize=7, rotation=45, ha='right')
        else:
            ax.text(0.5, 0.5, 'Insufficient data\nper study group', ha='center', va='center',
                    transform=ax.transAxes, fontsize=10, color='gray')

        ax.set_ylabel('log₁₀(Median UMI)', fontsize=10)
        ax.set_title('D. Library Depth by Study Group', fontsize=11, fontweight='bold', loc='left')
        ax.legend(handles=[Patch(facecolor=SNC_COLORS['Non-SnC'], label='Non-SnC'),
                           Patch(facecolor=SNC_COLORS['SnC'], label='SnC')],
                  fontsize=7, frameon=False, loc='upper right')
        style_ax(ax)

        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_depth_normalized.png', dpi=300, bbox_inches='tight')
        plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_depth_normalized.svg', dpi=300, bbox_inches='tight')
        plt.show()
        print("Saved depth-normalized CV figure (4 panels)")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# DEPTH-NORMALIZED CV — VIOLIN + SWARMPLOT
# ════════════════════════════════════════════════════════════════════════════════
#
# STYLE: Enclosed violin-swarm
#   - Full 4-sided frame, dark spines (#333), 0.8pt
#   - Heavy violin outlines (2.5pt, #333), transparent fill (alpha=0.25)
#   - Swarm dots overlaid, colored by covariate, white edge
#   - Paired lines faint behind dots
#   - Text stats top-left, no box, dark gray
#   - No gridlines, sans-serif, clean axis labels
#
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("DEPTH-NORMALIZED CV — VIOLIN + SWARMPLOT")
print("=" * 60)

from matplotlib.patches import Patch

study_groups = [g for g in GROUP_ORDER if g in depth_cv_df['Study_Group'].values]
SG_COLORS = {g: STUDY_GROUP_COLORS.get(g, '#808080') for g in study_groups}

donor_sg = paired_df.drop_duplicates('donor').set_index('donor')['Study_Group']

fig, axes = plt.subplots(1, 3, figsize=(13, 5))

def style_ax(ax):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)
        spine.set_color('#333')
    ax.tick_params(labelsize=9, width=0.8, length=4)

def darken_violins(ax):
    for collection in ax.collections:
        if hasattr(collection, 'set_edgecolor'):
            collection.set_edgecolor('#333')

# ═══════════════════════════════════════════════════════════════════════════════
# Panel A: Raw CV
# ═══════════════════════════════════════════════════════════════════════════════

ax = axes[0]
ax.grid(False)

raw_long = pd.DataFrame({
    'donor': list(pivot_raw.index) * 2,
    'Senescence': ['Non-SnC'] * len(pivot_raw) + ['SnC'] * len(pivot_raw),
    'CV': list(pivot_raw['Non-SnC']) + list(pivot_raw['SnC']),
})
raw_long['Study_Group'] = raw_long['donor'].map(donor_sg)

sns.violinplot(data=raw_long, x='Senescence', y='CV', order=['Non-SnC', 'SnC'],
               hue='Senescence', hue_order=['Non-SnC', 'SnC'],
               palette=SNC_COLORS, inner=None, linewidth=2.5, alpha=0.25, ax=ax, cut=0, legend=False)
darken_violins(ax)

for donor in pivot_raw.index:
    sg = donor_sg.get(donor, '')
    color = SG_COLORS.get(sg, '#808080')
    jx = np.random.uniform(-0.03, 0.03)
    ax.plot([0 + jx, 1 + jx], [pivot_raw.loc[donor, 'Non-SnC'], pivot_raw.loc[donor, 'SnC']],
            c=color, alpha=0.25, lw=0.5, zorder=1)

sns.swarmplot(data=raw_long, x='Senescence', y='CV', order=['Non-SnC', 'SnC'],
              hue='Study_Group', hue_order=study_groups, palette=SG_COLORS,
              size=7, edgecolor='white', linewidth=0.4, ax=ax, dodge=False, zorder=3)
ax.get_legend().remove()

sig_r = '***' if p_raw < 0.001 else '**' if p_raw < 0.01 else '*' if p_raw < 0.05 else 'ns'
n_down = (pivot_raw['SnC'] < pivot_raw['Non-SnC']).sum()
n_up = len(pivot_raw) - n_down
ax.text(0.03, 0.97, f'p={p_raw:.1e} {sig_r}\n{n_down}↓ {n_up}↑',
        transform=ax.transAxes, fontsize=8, ha='left', va='top', color='#333')

ax.set_xlabel(''); ax.set_ylabel('Median CV', fontsize=11)
ax.set_title('A. Raw CV', fontsize=12, fontweight='bold', loc='left')
style_ax(ax)

# ═══════════════════════════════════════════════════════════════════════════════
# Panel B: Depth-normalized CV
# ═══════════════════════════════════════════════════════════════════════════════

ax = axes[1]
ax.grid(False)

resid_long = pd.DataFrame({
    'donor': list(pivot_resid.index) * 2,
    'Senescence': ['Non-SnC'] * len(pivot_resid) + ['SnC'] * len(pivot_resid),
    'CV': list(pivot_resid['Non-SnC']) + list(pivot_resid['SnC']),
})
resid_long['Study_Group'] = resid_long['donor'].map(donor_sg)

sns.violinplot(data=resid_long, x='Senescence', y='CV', order=['Non-SnC', 'SnC'],
               hue='Senescence', hue_order=['Non-SnC', 'SnC'],
               palette=SNC_COLORS, inner=None, linewidth=2.5, alpha=0.25, ax=ax, cut=0, legend=False)
darken_violins(ax)

for donor in pivot_resid.index:
    sg = donor_sg.get(donor, '')
    color = SG_COLORS.get(sg, '#808080')
    jx = np.random.uniform(-0.03, 0.03)
    ax.plot([0 + jx, 1 + jx], [pivot_resid.loc[donor, 'Non-SnC'], pivot_resid.loc[donor, 'SnC']],
            c=color, alpha=0.25, lw=0.5, zorder=1)

sns.swarmplot(data=resid_long, x='Senescence', y='CV', order=['Non-SnC', 'SnC'],
              hue='Study_Group', hue_order=study_groups, palette=SG_COLORS,
              size=7, edgecolor='white', linewidth=0.4, ax=ax, dodge=False, zorder=3)
ax.get_legend().remove()

ax.axhline(0, color='k', linestyle='--', lw=0.6, alpha=0.3)

sig_d = '***' if p_resid < 0.001 else '**' if p_resid < 0.01 else '*' if p_resid < 0.05 else 'ns'
n_down = (pivot_resid['SnC'] < pivot_resid['Non-SnC']).sum()
n_up = len(pivot_resid) - n_down
ax.text(0.03, 0.97, f'p={p_resid:.1e} {sig_d}\n{n_down}↓ {n_up}↑',
        transform=ax.transAxes, fontsize=8, ha='left', va='top', color='#333')

ax.set_xlabel(''); ax.set_ylabel('Depth-normalized CV', fontsize=11)
ax.set_title('B. Corrected CV', fontsize=12, fontweight='bold', loc='left')
style_ax(ax)

# ═══════════════════════════════════════════════════════════════════════════════
# Panel C: ΔCV per donor
# ═══════════════════════════════════════════════════════════════════════════════

ax = axes[2]
ax.grid(False)

delta_per_donor = pivot_resid['SnC'] - pivot_resid['Non-SnC']

delta_df = pd.DataFrame({
    'donor': delta_per_donor.index,
    'delta_cv': delta_per_donor.values,
    'group': 'ΔCV',
})
delta_df['Study_Group'] = delta_df['donor'].map(donor_sg)

sns.violinplot(data=delta_df, x='group', y='delta_cv',
               color='#DDDDDD', inner=None, linewidth=2.5, alpha=0.35, ax=ax, cut=0)
darken_violins(ax)

sns.swarmplot(data=delta_df, x='group', y='delta_cv',
              hue='Study_Group', hue_order=study_groups, palette=SG_COLORS,
              size=8, edgecolor='white', linewidth=0.4, ax=ax, zorder=3)
ax.get_legend().remove()

ax.axhline(0, color='k', linestyle='--', lw=0.8, alpha=0.4)
ax.hlines(delta_per_donor.median(), -0.2, 0.2, color='k', linewidth=2.5, zorder=4)

from scipy.stats import wilcoxon as wilcoxon_1s
stat_delta, p_delta = wilcoxon_1s(delta_per_donor)
sig_delta = '***' if p_delta < 0.001 else '**' if p_delta < 0.01 else '*' if p_delta < 0.05 else 'ns'
n_neg = (delta_per_donor < 0).sum()
n_pos = (delta_per_donor >= 0).sum()

ax.text(0.03, 0.97, f'p={p_delta:.1e} {sig_delta}\nmed={delta_per_donor.median():.3f}\n{n_neg}↓ {n_pos}↑',
        transform=ax.transAxes, fontsize=8, ha='left', va='top', color='#333')

ax.set_xlabel(''); ax.set_ylabel('Depth-normalized ΔCV', fontsize=11)
ax.set_xticks([0])
ax.set_xticklabels(['SnC − Non-SnC'], fontsize=10)
ax.set_title('C. Per-donor ΔCV', fontsize=12, fontweight='bold', loc='left')
style_ax(ax)

# Legend
legend_handles = [Patch(facecolor=SG_COLORS[sg], label=sg, edgecolor='#333', linewidth=0.5)
                  for sg in study_groups]
fig.legend(handles=legend_handles, loc='lower center', fontsize=7, frameon=True,
           edgecolor='#333', ncol=len(study_groups), bbox_to_anchor=(0.5, -0.03))

plt.tight_layout()
plt.subplots_adjust(bottom=0.13)
plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_violin_swarm.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / f'{DATASET}_{CELL_TYPE.lower()}_cv_violin_swarm.svg', dpi=300, bbox_inches='tight')
plt.show()
print("Saved violin + swarmplot figure")

---
## 15 · Variance partitioning

**Why.** A different decomposition of the same phenomenon. Rather than asking
whether spread differs between arms, it asks where the total variance in
expression comes from: donor, senescence status, sex, cohort, residual.

**Why it matters beyond this module.** If the donor term dominates the
senescence term — which is the usual result in these data — that is the
quantitative statement behind the pseudoreplication rule applied throughout this
pipeline. Treating cells as independent replicates ignores the largest variance
component in the model.

**Formula.** Per gene, a mixed model with random effects for each grouping
factor; the reported quantity is each factor's share of total variance.

**Display.** Per-gene variance shares, then the distribution of shares across
genes per factor.

> **Language switch.** `variancePartition` is R-only, and the source ran this
> half under an R kernel. Here it runs through rpy2 `%%R` so the module stays
> one notebook. These cells read their own object from disk, so nothing crosses
> the Python/R boundary.

In [ ]:
# rpy2 bridge - sections 15 onward are R (variancePartition has no
# Python equivalent). One persistent R environment is shared across
# the %%R cells below.
%load_ext rpy2.ipython

In [ ]:
%%R
# ════════════════════════════════════════════════════════════════════════════════
# MODULE 08C: VARIANCE PARTITIONING ANALYSIS
# ════════════════════════════════════════════════════════════════════════════════
# Question: What factors explain variance in gene expression?
# Method: variancePartition on pseudobulk (donor × senescence)
# ════════════════════════════════════════════════════════════════════════════════

# ────────────────────────────────────────────────────────────────────────────────
# 08C-1: LIBRARIES
# ────────────────────────────────────────────────────────────────────────────────

suppressPackageStartupMessages({
  library(variancePartition)
  library(edgeR)
  library(ggplot2)
  library(dplyr)
  library(Seurat)
  library(qs)
})

cat("✓ Libraries loaded\n")

In [ ]:
%%R
# ────────────────────────────────────────────────────────────────────────────────
# 08C-2: CONFIGURATION
# ────────────────────────────────────────────────────────────────────────────────

DATASET <- "psychad_aging"
CELL_TYPE <- "OPC"

STUDY_CONFIG <- list(
  psychad_aging = list(
    type = "aging",
    cell_type_col = "subclass",
    sample_col = "Sample",
    senescence_col = "senescence_label",
    covariates = list(
      Donor = "Sample",
      Senescence = "senescence_label",
      Study_Group = "Study_Group",
      Sex = "Sex",
      Cohort = "Cohort",
      Log_Library_Depth = "Log_Library_Depth" 
    ),
    group_order = c("Age_20_29", "Age_30_39", "Age_40_49", "Age_50_59",
                    "Age_60_69", "Age_70_79", "Age_80_100")
  ),
  psychad_ad = list(
    type = "disease",
    cell_type_col = "subclass",
    sample_col = "Sample",
    senescence_col = "senescence_label",
    covariates = list(
      Donor = "Sample",
      Senescence = "senescence_label",
      Study_Group = "Study_Group",
      Sex = "Sex",
      Cohort = "Cohort",
      Log_Library_Depth = "Log_Library_Depth"  
    ),
    group_order = c("Control", "MCI", "AD")
  ),
  psychencode = list(
    type = "aging",
    cell_type_col = "cell_type",
    sample_col = "sample_id",
    senescence_col = "senescence_label",
    covariates = list(
      Donor = "sample_id",
      Senescence = "senescence_label",
      Study_Group = "Study_Group",
      Sex = "Biological_Sex",
      Cohort = "Cohort",
      Log_Library_Depth = "Log_Library_Depth" 
    ),
    group_order = c("Age_20_29", "Age_30_39", "Age_40_49", "Age_50_59",
                    "Age_60_69", "Age_70_79", "Age_80_100")
  )
)

# Extract config
config <- STUDY_CONFIG[[DATASET]]
CELL_TYPE_COL <- config$cell_type_col
SAMPLE_COL <- config$sample_col
SENESCENCE_COL <- config$senescence_col
COVARIATES <- config$covariates

# Paths
BASE_DIR <- SEN_DATA
SC_FILE <- file.path(BASE_DIR, "data", "04_subsetting", DATASET, paste0(DATASET, "_subclustered.qs"))
RESULTS_DIR <- file.path(BASE_DIR, "results", "08_variability", DATASET, tolower(CELL_TYPE))
FIGURES_DIR <- file.path(BASE_DIR, "figures", "08_variability", DATASET, tolower(CELL_TYPE))

dir.create(RESULTS_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(FIGURES_DIR, recursive = TRUE, showWarnings = FALSE)

# Colors
VP_COLORS <- c(
  "Donor" = "#4E79A7",
  "Senescence" = "#D73027",
  "Study_Group" = "#59A14F",
  "Sex" = "#F28E2B",
  "Cohort" = "#B07AA1",
  "Log_Library_Depth" = "#76B7B2",
  "Residuals" = "#BDBDBD"
)  

cat("════════════════════════════════════════════════════════════════════════════════\n")
cat("Dataset:", DATASET, "\n")
cat("Cell type:", CELL_TYPE, "\n")
cat("Covariates:", paste(names(COVARIATES), collapse = ", "), "\n")
cat("════════════════════════════════════════════════════════════════════════════════\n")

In [ ]:
%%R
# ────────────────────────────────────────────────────────────────────────────────
# 08C-3: LOAD DATA
# ────────────────────────────────────────────────────────────────────────────────

cat("\nLoading data...\n")

seurat_all <- qread(SC_FILE)
cat("  Total cells:", ncol(seurat_all), "\n")

seurat <- subset(seurat_all, subset = !!sym(CELL_TYPE_COL) == CELL_TYPE)
cat(" ", CELL_TYPE, "cells:", ncol(seurat), "\n")

cat("\nSenescence distribution:\n")
print(table(seurat@meta.data[[SENESCENCE_COL]]))

rm(seurat_all)
gc()

cat("\n✓ Data loaded\n")

In [ ]:
%%R
seurat

In [ ]:
%%R
# ────────────────────────────────────────────────────────────────────────────────
# 08C-4: CREATE PSEUDOBULK
# ────────────────────────────────────────────────────────────────────────────────

cat("\n")
cat("════════════════════════════════════════════════════════════════════════════════\n")
cat("Creating pseudobulk...\n")
cat("════════════════════════════════════════════════════════════════════════════════\n")

counts <- GetAssayData(seurat, layer = "counts")

seurat$donor_snc <- paste0(seurat@meta.data[[SAMPLE_COL]], "-", 
                           seurat@meta.data[[SENESCENCE_COL]])

donor_snc_ids <- unique(seurat$donor_snc)
cat("  Aggregating", length(donor_snc_ids), "samples...\n")

pseudobulk <- sapply(donor_snc_ids, function(id) {
  cells <- which(seurat$donor_snc == id)
  if (length(cells) > 1) {
    Matrix::rowSums(counts[, cells])
  } else {
    counts[, cells]
  }
})
colnames(pseudobulk) <- donor_snc_ids

cat("  Pseudobulk:", nrow(pseudobulk), "genes x", ncol(pseudobulk), "samples\n")

In [ ]:
%%R
# ────────────────────────────────────────────────────────────────────────────────
# 08C-5: CREATE METADATA
# ────────────────────────────────────────────────────────────────────────────────
cat("\nCreating metadata...\n")

# Build summarise expressions dynamically (exclude Log_Library_Depth — computed from pseudobulk)
meta_covariates <- COVARIATES[names(COVARIATES) != "Log_Library_Depth"]

summarise_exprs <- lapply(names(meta_covariates), function(out_name) {
  src_col <- meta_covariates[[out_name]]
  rlang::expr(first(.data[[!!src_col]]))
})

names(summarise_exprs) <- names(meta_covariates)

summarise_exprs$n_cells <- rlang::expr(n())

meta <- seurat@meta.data %>%
  mutate(donor_snc = paste0(.data[[SAMPLE_COL]], "-", .data[[SENESCENCE_COL]])) %>%
  group_by(donor_snc) %>%
  summarise(!!!summarise_exprs, .groups = "drop") %>%
  as.data.frame()

rownames(meta) <- meta$donor_snc

cat("  Samples:", nrow(meta), "\n")

# Check NAs
na_check <- sapply(meta[, names(meta_covariates), drop = FALSE], function(x) sum(is.na(x)))
if (any(na_check > 0)) {
  cat("  ⚠ NAs found:", paste(names(na_check)[na_check > 0], na_check[na_check > 0], sep = "=", collapse = ", "), "\n")
} else {
  cat("  ✓ No missing values\n")
}

In [ ]:
%%R
# ────────────────────────────────────────────────────────────────────────────────
# 08C-6: FILTER AND NORMALIZE
# ────────────────────────────────────────────────────────────────────────────────
cat("\nFiltering and normalizing...\n")

common_samples <- intersect(colnames(pseudobulk), rownames(meta))

pseudobulk <- pseudobulk[, common_samples]

meta <- meta[common_samples, ]

cat("  Aligned samples:", length(common_samples), "\n")

# Add Log_Library_Depth (computed from pseudobulk, not from Seurat metadata)
meta$Log_Library_Depth <- log10(colSums(pseudobulk))

cat("  Log_Library_Depth range:", round(min(meta$Log_Library_Depth), 2), "-",
    round(max(meta$Log_Library_Depth), 2), "\n")

dge <- DGEList(counts = pseudobulk)

keep <- filterByExpr(dge, min.count = 10, min.total.count = 15)

dge <- dge[keep, ]

cat("  Genes after filtering:", nrow(dge), "\n")

dge <- calcNormFactors(dge, method = "TMM")

logCPM <- cpm(dge, log = TRUE, prior.count = 1)

cat("  Final:", nrow(logCPM), "genes x", ncol(logCPM), "samples\n")

# Verify alignment
stopifnot(all(colnames(logCPM) == rownames(meta)))

cat("  ✓ Alignment verified\n")

In [ ]:
%%R
# ────────────────────────────────────────────────────────────────────────────────
# 08C-7: RUN VARIANCE PARTITIONING
# ────────────────────────────────────────────────────────────────────────────────
cat("\n")
cat("════════════════════════════════════════════════════════════════════════════════\n")
cat("Running variance partitioning...\n")
cat("════════════════════════════════════════════════════════════════════════════════\n")

# Convert categorical covariates to factors (NOT Log_Library_Depth)
categorical_covs <- setdiff(names(COVARIATES), "Log_Library_Depth")
for (cov in categorical_covs) {
  meta[[cov]] <- as.factor(meta[[cov]])
}

# Ensure Log_Library_Depth is numeric
meta$Log_Library_Depth <- as.numeric(meta$Log_Library_Depth)

cat("\nFactor levels:\n")
for (cov in categorical_covs) {
  n_lev <- nlevels(meta[[cov]])
  if (n_lev <= 5) {
    cat("  ", cov, ": ", paste(levels(meta[[cov]]), collapse = ", "), "\n", sep = "")
  } else {
    cat("  ", cov, ": ", n_lev, " levels\n", sep = "")
  }
}
cat("  Log_Library_Depth: continuous (range ",
    round(min(meta$Log_Library_Depth), 2), " - ",
    round(max(meta$Log_Library_Depth), 2), ")\n", sep = "")

# Build formula: random effects for categorical, fixed for continuous
random_covs <- categorical_covs
fixed_covs <- c("Log_Library_Depth")

form_str <- paste0("~ ",
  paste0("(1|", random_covs, ")", collapse = " + "),
  " + ",
  paste0(fixed_covs, collapse = " + ")
)
form <- as.formula(form_str)
cat("\nFormula:", form_str, "\n")
cat("Genes:", nrow(logCPM), "\n")
cat("Samples:", ncol(logCPM), "\n")

cat("\nRunning... (this may take several minutes)\n")
varPart <- fitExtractVarPartModel(logCPM, form, meta)
cat("\n✓ Variance partitioning complete\n")

In [ ]:
%%R
# ────────────────────────────────────────────────────────────────────────────────
# 08C-8: RESULTS
# ────────────────────────────────────────────────────────────────────────────────

cat("\n")
cat("════════════════════════════════════════════════════════════════════════════════\n")
cat("Results\n")
cat("════════════════════════════════════════════════════════════════════════════════\n")

varPart_summary <- data.frame(
  Factor = colnames(varPart),
  Mean_pct = round(colMeans(varPart) * 100, 2),
  Median_pct = round(apply(varPart, 2, median) * 100, 2),
  SD_pct = round(apply(varPart, 2, sd) * 100, 2)
)
varPart_summary <- varPart_summary[order(-varPart_summary$Mean_pct), ]

print(varPart_summary, row.names = FALSE)

In [ ]:
%%R
# ────────────────────────────────────────────────────────────────────────────────
# 08C-9: VISUALIZATION
# ────────────────────────────────────────────────────────────────────────────────

cat("\nCreating figures...\n")

# Violin plot
pdf(file.path(FIGURES_DIR, paste0(DATASET, "_", tolower(CELL_TYPE), "_varpart_violin.pdf")),
    width = 6, height = 4)
plotVarPart(varPart, col = VP_COLORS[colnames(varPart)])
dev.off()
cat("  ✓ varpart_violin.svg\n")

# Bar plot
p <- ggplot(varPart_summary, aes(x = reorder(Factor, -Mean_pct), y = Mean_pct, fill = Factor)) +
  geom_bar(stat = "identity", width = 0.7) +
  geom_errorbar(aes(ymin = Mean_pct - SD_pct, ymax = Mean_pct + SD_pct), width = 0.2) +
  scale_fill_manual(values = VP_COLORS) +
  labs(x = NULL, y = "Variance Explained (%)") +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    panel.grid.major.x = element_blank(),
    legend.position = "none"
  )

ggsave(file.path(FIGURES_DIR, paste0(DATASET, "_", tolower(CELL_TYPE), "_varpart_bar.svg")),
       p, width = 5, height = 4)
cat("  ✓ varpart_bar.svg\n")

In [ ]:
%%R
# ────────────────────────────────────────────────────────────────────────────────
# 08C-10: SAVE
# ────────────────────────────────────────────────────────────────────────────────

cat("\nSaving results...\n")

write.csv(as.data.frame(varPart), 
          file.path(RESULTS_DIR, paste0(DATASET, "_", tolower(CELL_TYPE), "_varpart_full.csv")))
write.csv(varPart_summary, 
          file.path(RESULTS_DIR, paste0(DATASET, "_", tolower(CELL_TYPE), "_varpart_summary.csv")),
          row.names = FALSE)

cat("  ✓ varpart_full.csv\n")
cat("  ✓ varpart_summary.csv\n")

# ────────────────────────────────────────────────────────────────────────────────
# 08C-11: SUMMARY
# ────────────────────────────────────────────────────────────────────────────────

cat("\n")
cat("════════════════════════════════════════════════════════════════════════════════\n")
cat("08C Complete\n")
cat("════════════════════════════════════════════════════════════════════════════════\n")
cat("\nVariance explained:\n")
for (i in 1:nrow(varPart_summary)) {
  cat("  ", varPart_summary$Factor[i], ": ", varPart_summary$Mean_pct[i], "%\n", sep = "")
}

---
## Reading this module

**The balance and depth checks come first for a reason.** Section 03 measures a
bias that inflates senescent-cell CV; section 05 measures one that deflates it.
Neither is negligible at 2-5% prevalence and 1.9× depth, and they act in
opposite directions — so the net bias cannot be signed from first principles.
Any CV difference should be reported alongside both numbers.

**The non-DEG, non-HVG stratum is the cleanest read.** Those genes were neither
selected for variance nor found to have shifted means. A CV difference there is
the one hardest to explain as a side effect.

**Variance partitioning answers a question the CV analysis cannot** — not
whether senescent cells are noisier, but whether senescence is a large term at
all relative to donor identity.